# VQ-VAE-2 Latent Feature Extraction

Extract and visualize latent representations from all 3 hierarchical levels of the VQ-VAE-2 encoder.

The content/style Gumbel mask is applied **after** the codebook projection (`conv_in`),
directly on the `embed_dim`-sized representation.  This notebook accounts for that by:
- Building the model with `content_size` and `style_size` (required to reconstruct `channel_logits`)
- Using `pool_only=True` to get pooled `(B, embed_dim)` vectors per level (memory-efficient)
- Using `estimated_content_indices` returned by the forward pass to split content vs style dims

In [ ]:
import sys
import os

# Add project root to sys.path (this notebook lives in eval/)
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import json

# Import local modules
import models.vqvae as vqvae
import utils.utils as utils
from utils.utils import load_data, CreateBrainMaskd, ApplyBrainMaskd

# ============================================================
# CONFIGURATION
# ============================================================
# Use vqvae_best.pt for the best checkpoint (by rolling avg total loss),
# or vqvae_model.pt for the latest checkpoint.
CHECKPOINT_PATH = (
    "/home/ng24/projects/multiview-crl/results/ADNI_registered/multiview-06-content-lr-002-single-level/vqvae_best1.pt"
)
DATA_DIR = "/data/natalia/ADNI_stripped"
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
NUM_SAMPLES = 200  # number of subjects to process
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ADNI 2-view split (must match training)
CONTENT_SIZE = 256  # dimensions treated as content (shared between T1/T2)
STYLE_SIZE = 256  # dimensions treated as style (view-specific)

# Resampling: "original" (1 mm, ~182x218x182) or "2mm" (91x109x91)
RESAMPLE_MODE = "original"
# ============================================================

print(f"Using device: {DEVICE}")

## 1. Load checkpoint & build model

In [ ]:
# Load settings saved alongside the checkpoint
settings_path = os.path.join(os.path.dirname(CHECKPOINT_PATH), "settings.json")
with open(settings_path, "r") as f:
    settings = json.load(f)

print("Model settings:")
for k in [
    "vqvae_hidden_channels",
    "vqvae_res_channels",
    "vqvae_nb_res_layers",
    "vqvae_nb_levels",
    "vqvae_embed_dim",
    "vqvae_nb_entries",
    "vqvae_scaling_rates",
]:
    print(f"  {k}: {settings.get(k, 'N/A')}")

# Override CONTENT_SIZE / STYLE_SIZE from saved settings.
# settings.json stores content_indices (list[list[int]]) and style_indices (list[int])
# written by update_args() at training time — use them so the VQVAE is rebuilt
# with exactly the same content/style ratio that was used during training.
# Falling back to the hardcoded values only when the keys are absent (old checkpoints).
if "content_indices" in settings and "style_indices" in settings:
    CONTENT_SIZE = len(settings["content_indices"][0])
    STYLE_SIZE = len(settings["style_indices"])
    print(f"\n  content_size : {CONTENT_SIZE}  (from settings.json['content_indices'])")
    print(f"  style_size   : {STYLE_SIZE}  (from settings.json['style_indices'])")
else:
    print(f"\n  ⚠ content_indices / style_indices not found in settings.json.")
    print(f"    Using hardcoded CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}.")
    print(f"    If these don't match the training run, content_channels will be wrong.")

In [ ]:
# Build model — must match the exact architecture used during training.
# inject_style_to_decoder controls whether decoders[0] has a final_conv branch;
# if it differs from the checkpoint the decoder weights will silently not load.
inject_style = settings.get("inject_style_to_decoder", False)

# ── Auto-detect content/style split from checkpoint weights ──────────────────
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint["encoders"]

# Strip MoCo "online." prefix if present to find the right keys
_prefix = ""
if any(k.startswith("online.") for k in state_dict):
    _prefix = "online."

# channel_logits is now a ParameterDict keyed by level.
# Old checkpoints: "module.channel_logits" (single param)
# New checkpoints: "module.channel_logits.0", "module.channel_logits.1", etc.
_cl_key_old = f"{_prefix}module.channel_logits"

# Find all channel_logits.X keys where X is a digit
_cl_levels = []
for k in state_dict:
    stripped = k[len(_prefix) :] if k.startswith(_prefix) else k
    if stripped.startswith("module.channel_logits."):
        suffix = stripped[len("module.channel_logits.") :]
        if suffix.isdigit():
            _cl_levels.append(int(suffix))
_cl_levels = sorted(_cl_levels)

# ── Detect mask_mode from checkpoint keys ────────────────────────────────────
# fixed mode: has fixed_mask_* buffers, no channel_logits
# learned mode: has channel_logits params
# onthefly mode: neither (logits computed from activations at runtime)
_has_fixed_mask = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.fixed_mask_") for k in state_dict
)

if _has_fixed_mask:
    _detected_mask_mode = "fixed"
elif _cl_levels or _cl_key_old in state_dict:
    _detected_mask_mode = "learned"
else:
    _detected_mask_mode = "onthefly"

# Allow settings.json to override (it's always authoritative if present)
_mask_mode = settings.get("mask_mode", _detected_mask_mode)

if _cl_levels:
    # New-style per-level channel_logits
    _cl_key_0 = f"{_prefix}module.channel_logits.{_cl_levels[0]}"
    _mask_dim = state_dict[_cl_key_0].shape[0]
    content_style_levels = _cl_levels
elif _cl_key_old in state_dict:
    # Old-style single channel_logits
    _mask_dim = state_dict[_cl_key_old].shape[0]
    content_style_levels = [0]
else:
    _mask_dim = None
    # No channel_logits (onthefly or fixed mode) — read content_style_levels from settings
    content_style_levels = settings.get("content_style_levels", [0])

# ── Per-level content_channels detection from codebook input sizes ────────────
hidden_channels = settings["vqvae_hidden_channels"]
_embed_dim = settings["vqvae_embed_dim"]
_nb_levels = settings["vqvae_nb_levels"]

# Detect per-level content_channels from codebook conv_in dimensions.
# Works for any mask_mode (learned, onthefly, or fixed) as long as the codebook was
# sized for the content subset (content_channels < hidden_channels).
_content_ch_per_level = {}
_used_codebook_detection = False
content_ratios = None

for lvl in content_style_levels:
    _cb_key = f"{_prefix}module.codebooks.{lvl}.conv_in.weight"
    if _cb_key in state_dict:
        _cb_in = state_dict[_cb_key].shape[1]
        if lvl == _nb_levels - 1:
            # Top level: codebook input = content_channels (no embed_dim concat)
            _content_ch_per_level[lvl] = _cb_in
        else:
            # Lower levels: codebook input = content_channels + embed_dim
            _content_ch_per_level[lvl] = _cb_in - _embed_dim

# Check if codebook detection is valid: if ALL detected levels give
# content_channels == hidden_channels, the checkpoint predates per-level
# codebook sizing — fall back to settings-based CONTENT_SIZE / STYLE_SIZE.
_all_full_width = all(cc == hidden_channels for cc in _content_ch_per_level.values()) if _content_ch_per_level else True

if not _all_full_width:
    # Codebook was sized for content masking — use detected values
    _used_codebook_detection = True

    _first_lvl = content_style_levels[0]
    _content_channels = _content_ch_per_level.get(_first_lvl, hidden_channels)
    _style_channels = hidden_channels - _content_channels

    CONTENT_SIZE = _content_channels
    STYLE_SIZE = _style_channels

    # Compute per-level content_ratios if levels have different content sizes
    _ratios = [_content_ch_per_level[lvl] / hidden_channels for lvl in content_style_levels]
    if len(set(_ratios)) > 1:
        content_ratios = _ratios

    _mode_desc = {
        "learned": "learned",
        "onthefly": "onthefly (no channel_logits)",
        "fixed": "fixed (first K = content)",
    }
    print(f"Auto-detected from checkpoint (per-level codebook sizing):")
    print(f"  mask_mode             : {_mode_desc.get(_mask_mode, _mask_mode)}")
    print(f"  content_style_levels  : {content_style_levels}")
    for lvl in content_style_levels:
        cc = _content_ch_per_level.get(lvl, "?")
        print(f"  level {lvl} content_ch    : {cc} / {hidden_channels}  ({cc/hidden_channels:.1%})")
    print(f"  -> CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}")
    if content_ratios:
        print(f"  -> content_ratios={content_ratios}")
else:
    # Old checkpoint: codebooks use full hidden_channels.
    # Keep CONTENT_SIZE / STYLE_SIZE from settings cell (cell above).
    print(f"Codebook detection gives content_ch == hidden_channels at all levels.")
    print(f"  content_style_levels  : {content_style_levels}")
    print(f"  → Using CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE} from settings.json")

# ── Detect whether checkpoint uses hidden_channels or embed_dim masking ──────
if _mask_dim is not None and _mask_dim != hidden_channels:
    raise RuntimeError(
        f"Checkpoint has channel_logits of size {_mask_dim} but hidden_channels={hidden_channels}. "
        f"This checkpoint was likely trained with an older code version where the mask was on "
        f"embed_dim={_embed_dim}. You need to checkout the matching code version "
        f"to load this checkpoint."
    )

# ── Detect separate_encoders from checkpoint keys ────────────────────────────
# If the checkpoint contains "module.encoders_v1.*" keys, the model was trained
# with --separate-encoders.
_sep_enc = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.encoders_v1.") for k in state_dict
)
if _sep_enc:
    print("Detected separate_encoders in checkpoint (encoders_v1 keys found).")

_style_inj_mode = settings.get("style_injection_mode", "concat")

# ── Read training flags that affect the encoder forward pass ─────────────────
# These MUST match training exactly, otherwise the encoder produces different
# features at inference time (e.g. style channels zeroed between levels when
# they should be passed through, causing out-of-distribution inputs at higher levels).
_pass_full = settings.get("pass_full_to_next_level", False)
_narrow_enc = settings.get("narrow_encoder_input", False)
_top_recon = settings.get("top_level_recon_only", False)
_quantize_style = settings.get("quantize_style", False)
_style_embed_dim = settings.get("style_embed_dim", None)
_style_nb_entries_raw = settings.get("style_nb_entries", None)
_use_content_proj = settings.get("use_content_projection", False)

# ── Normalize per-level codebook sizes ───────────────────────────────────────
# --vqvae-nb-entries / --style-nb-entries are argparse nargs="+" → always a
# list in settings.json. W&B sweeps occasionally round-trip these as
# stringified lists ("[384]") or nested lists ([[384]]), which then reach
# CodeLayer where torch.randn(embed_dim, nb_entries) raises
#   TypeError: randn(): argument 'size' failed to unpack ... got list
# Flatten + coerce to int / flat list[int] before handing to VQVAE.
import ast as _ast


def _normalize_entries(v):
    if v is None:
        return None
    if isinstance(v, str):
        try:
            v = _ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return int(v)
    if isinstance(v, (list, tuple)):
        flat = []
        for x in v:
            if isinstance(x, (list, tuple)):
                flat.extend(x)
            else:
                flat.append(x)
        flat = [int(x) for x in flat]
        return flat[0] if len(flat) == 1 else flat
    return int(v)


_nb_entries = _normalize_entries(settings["vqvae_nb_entries"])
_style_nb_entries = _normalize_entries(_style_nb_entries_raw)
print(f"vqvae_nb_entries (normalized) : {_nb_entries}")
if _style_nb_entries is not None:
    print(f"style_nb_entries (normalized) : {_style_nb_entries}")

# ── Optional: skip levels in the final reconstruction decoder ────────────────
# Define SKIP_RECON_LEVELS in a cell above to ablate finer codes, e.g.
#     SKIP_RECON_LEVELS = [0, 1]
# zeros out levels 0 and 1 in the final (level-0) decoder concat, leaving only
# the top (coarsest) codes to drive reconstruction. The top level itself cannot
# be skipped. If undefined, falls back to whatever was stored at training time.
try:
    _skip_recon = SKIP_RECON_LEVELS  # noqa: F821 — optional user override
    print(f"⚠ Notebook override: skip_decoder_concat_levels={_skip_recon}")
except NameError:
    _skip_recon = settings.get("skip_decoder_concat_levels", None)
    if _skip_recon:
        print(f"skip_decoder_concat_levels (from settings): {_skip_recon}")

vqvae_model = vqvae.VQVAE(
    in_channels=1,
    hidden_channels=hidden_channels,
    res_channels=settings["vqvae_res_channels"],
    nb_res_layers=settings.get("vqvae_nb_res_layers", 2),
    nb_levels=_nb_levels,
    embed_dim=_embed_dim,
    nb_entries=_nb_entries,
    scaling_rates=settings["vqvae_scaling_rates"],
    content_size=CONTENT_SIZE,
    style_size=STYLE_SIZE,
    inject_style_to_decoder=inject_style,
    content_style_levels=content_style_levels,
    content_ratios=content_ratios,
    separate_encoders=_sep_enc,
    mask_mode=_mask_mode,
    style_injection_mode=_style_inj_mode,
    pass_full_to_next_level=_pass_full,
    narrow_encoder_input=_narrow_enc,
    top_level_recon_only=_top_recon,
    quantize_style=_quantize_style,
    style_embed_dim=_style_embed_dim,
    style_nb_entries=_style_nb_entries,
    use_content_projection=_use_content_proj,
    skip_decoder_concat_levels=_skip_recon,
)

content_channels = vqvae_model.content_channels
print(f"\nhidden_channels         : {hidden_channels}")
print(f"content_channels        : {content_channels}")
if vqvae_model.content_channels_per_level:
    print(f"content_channels_per_lvl: {vqvae_model.content_channels_per_level}")
print(f"content_style_levels    : {content_style_levels}")
print(f"mask_mode               : {_mask_mode}")
print(f"inject_style_to_decoder : {inject_style}")
print(f"separate_encoders       : {_sep_enc}")
print(f"style_injection_mode    : {_style_inj_mode}")
print(f"pass_full_to_next_level : {_pass_full}")
print(f"narrow_encoder_input    : {_narrow_enc}")
print(f"top_level_recon_only    : {_top_recon}")
print(f"quantize_style          : {_quantize_style}")
print(f"use_content_projection  : {_use_content_proj}")

# Wrap in DataParallel to match the saved checkpoint structure
vqvae_model = torch.nn.DataParallel(vqvae_model, device_ids=[0])

# ── Key remapping ─────────────────────────────────────────────────────────────
new_state_dict = {}
is_moco_checkpoint = any(k.startswith("online.") for k in state_dict)
if is_moco_checkpoint:
    print("Detected MoCoEncoder checkpoint — stripping 'online.' prefix.")

for k, v in state_dict.items():
    k = k.replace(".blocks.", ".stack.")  # legacy rename
    # Migrate old single channel_logits → ParameterDict key "0"
    if k.endswith("module.channel_logits") and not any(c.isdigit() for c in k.split(".")[-1]):
        k = k + ".0"
    if k.startswith("online."):
        new_state_dict[k[len("online.") :]] = v
    elif k.startswith(
        ("momentum_encoders.", "momentum_codebook_projs.", "momentum_content_proj.", "momentum_level0_proj.", "queue_")
    ):
        pass  # MoCo-only state — discard
    elif k.startswith("momentum_encoders_v1."):
        pass  # MoCo momentum copy of view-1 encoder — discard
    else:
        new_state_dict[k] = v

# ── Drop keys with shape mismatches (e.g. projection head from older config) ─
_model_sd = vqvae_model.state_dict()
_shape_skipped = []
for k in list(new_state_dict):
    if k in _model_sd and new_state_dict[k].shape != _model_sd[k].shape:
        _shape_skipped.append(f"{k}: ckpt {new_state_dict[k].shape} vs model {_model_sd[k].shape}")
        del new_state_dict[k]
if _shape_skipped:
    print(f"\u26a0 Skipped {len(_shape_skipped)} keys with shape mismatch:")
    for s in _shape_skipped:
        print(f"    {s}")

missing, unexpected = vqvae_model.load_state_dict(new_state_dict, strict=False)

missing_real = [k for k in missing if "num_batches_tracked" not in k]
unexpected_real = [k for k in unexpected if "num_batches_tracked" not in k]

if missing_real:
    print(f"\u26a0 Missing keys ({len(missing_real)}): {missing_real[:6]}{'...' if len(missing_real) > 6 else ''}")
if unexpected_real:
    print(
        f"\u26a0 Unexpected keys ({len(unexpected_real)}): {unexpected_real[:6]}{'...' if len(unexpected_real) > 6 else ''}"
    )
if not missing_real and not unexpected_real:
    print("\u2713 Checkpoint loaded cleanly")
elif not missing_real:
    print("\u2713 Checkpoint loaded (unexpected keys are harmless extras in the file)")

vqvae_model.to(DEVICE)
vqvae_model.eval()
print(f"\u2713 Model on {DEVICE}, step {checkpoint['step']}")
print(f"  Total params: {sum(p.numel() for p in vqvae_model.parameters()):,}")

In [ ]:
# -- Diagnostic: isolate where the constant-output problem lives --
import torch

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# 1. Check encoder weight statistics (are weights non-trivial?)
print("=== Encoder weight check ===")
for name, p in list(vqvae_module.encoders[0].named_parameters())[:6]:
    print(
        f"  {name}: shape={tuple(p.shape)}  "
        f"mean={p.data.mean():.6f}  std={p.data.std():.6f}  "
        f"absmax={p.data.abs().max():.6f}"
    )

# 2. Check ReZero alpha values (if 0, residual blocks are identity)
print("\n=== ReZero alpha values ===")
for name, p in vqvae_module.encoders[0].named_parameters():
    if "alpha" in name:
        print(f"  {name}: {p.data.item():.6f}")

# 3. Test raw encoder (bypass VQVAE forward)
print("\n=== Raw encoder test ===")
with torch.no_grad():
    d1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    d2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)

    # Run through encoder level 0 directly
    out1 = vqvae_module.encoders[0](d1)
    out2 = vqvae_module.encoders[0](d2)

    print(f"  Encoder output shape: {out1.shape}")
    print(f"  out1 range: [{out1.min():.4f}, {out1.max():.4f}]  mean={out1.mean():.6f}")
    print(f"  out2 range: [{out2.min():.4f}, {out2.max():.4f}]  mean={out2.mean():.6f}")

    # Check spatial variation (different voxels should have different values)
    print(f"  out1 spatial std (per-channel): {out1[0].std(dim=[1, 2, 3]).mean():.6f}")

    # Pool and compare
    p1 = out1.mean(dim=[2, 3, 4])
    p2 = out2.mean(dim=[2, 3, 4])
    cos = torch.nn.functional.cosine_similarity(p1, p2).item()
    print(f"  Pooled cosine similarity: {cos:.6f}")
    if cos > 0.999:
        print("  WARNING: Raw encoder also produces constant output!")
    else:
        print("  OK: Raw encoder works - bug is in VQVAE.forward()")

# 4. Test layer by layer to find where signal dies
print("\n=== Layer-by-layer signal propagation ===")
with torch.no_grad():
    x1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    x2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    for layer_idx, layer in enumerate(vqvae_module.encoders[0].layers):
        x1 = layer(x1)
        x2 = layer(x2)
        cos = torch.nn.functional.cosine_similarity(x1.flatten(1), x2.flatten(1)).item()
        print(
            f"  After layer {layer_idx} ({type(layer).__name__}): "
            f"shape={tuple(x1.shape)}  cos={cos:.6f}  "
            f"range=[{x1.min():.3f},{x1.max():.3f}]"
        )

## 2. Load dataset & transforms

In [ ]:
df = pd.read_csv(CSV_PATH)
label_values = sorted(df["Group"].unique())
label_map = {v: i for i, v in enumerate(label_values)}
label_names = {i: v for v, i in label_map.items()}
print(f"Labels: {label_map}")

items, missing = load_data(df, DATA_DIR, label_map)
print(f"Loaded {len(items)} samples, {len(missing)} missing files")
items = items[:NUM_SAMPLES]
print(f"Using first {len(items)} subjects")

In [ ]:
# Match the training val pipeline EXACTLY. Previously the notebook always
# applied CreateBrainMaskd (intensity > 0 threshold). Training with masks on
# disk skips that and loads the proper brain-mask .nii.gz directly via
# LoadImaged — a different mask → different NormalizeIntensityd stats →
# different features → different modality probe result. Fix: delegate to
# utils.utils.transforms and use masks_from_disk when the CSV items carry
# mask paths.
from utils.utils import transforms as get_transforms

# Read spacing and spatial_size from settings.json (saved at training time)
spacing = settings.get("image_spacing", 2.0)
crop_margin = settings.get("crop_margin", 0)

_saved_spatial = settings.get("spatial_size", None)
if _saved_spatial is not None:
    spatial_size = tuple(_saved_spatial)
elif spacing == 1.0:
    spatial_size = (182, 218, 182)
else:
    spatial_size = (91, 109, 91)

# Mirror data/datasets.py:1283 — masks_from_disk is True iff every loaded
# item carries both mask paths.
masks_from_disk = all(("mask_image" in it) and ("mask_z_image" in it) for it in items)

_, _val_transforms_raw = get_transforms(
    spacing=spacing,
    crop_margin=crop_margin,
    spatial_size=spatial_size,
    masks_from_disk=masks_from_disk,
    asymmetric_aug=False,
)

# The raw val transform expects:
#   - ``mask_t1``/``mask_t2`` paths when ``masks_from_disk=True`` (loaded by LoadImaged)
#   - a ``label`` key at the end (required by the final ToTensord in utils.utils.transforms)
# Existing notebook cells pass only ``image_t1``/``image_t2``. Wrap the
# transform to auto-inject mask paths and a placeholder label, keyed on the
# T1 image path.
_img_to_extras = {
    it["image"]: {
        "mask_t1": it.get("mask_image"),
        "mask_t2": it.get("mask_z_image"),
        "label": it.get("label", 0),
    }
    for it in items
}


def val_transforms(data_dict):
    d = dict(data_dict)
    extras = _img_to_extras.get(d.get("image_t1"), {})
    if masks_from_disk:
        if "mask_t1" not in d and extras.get("mask_t1") is not None:
            d["mask_t1"] = extras["mask_t1"]
        if "mask_t2" not in d and extras.get("mask_t2") is not None:
            d["mask_t2"] = extras["mask_t2"]
    if "label" not in d:
        d["label"] = extras.get("label", 0)
    return _val_transforms_raw(d)


print(f"Transforms ready — spacing={spacing}mm, spatial_size={spatial_size}, " f"masks_from_disk={masks_from_disk}")
if not masks_from_disk:
    print(
        "  (No disk masks found on items — using intensity-threshold CreateBrainMaskd.\n"
        "  If training ran with disk masks, the modality probe will differ from the training log.)"
    )

## 3. Feature extraction

We call `vqvae_model(img, pool_only=True, return_recon=False)` which returns:
- `encoder_features`: list of `(B, embed_dim)` pooled vectors, one per level
  - All levels are pooled from the codebook projection (embed_dim space)
  - When `channel_logits` is active, content/style split is in this embed_dim space
- `estimated_content_indices`: the Gumbel-selected embedding dim indices at level 0

For visualisation we split level-0 pooled features into **content** and **style** subsets
using `estimated_content_indices`.

In [ ]:
nb_levels = settings["vqvae_nb_levels"]

# Storage
all_features = {f"level_{i}": [] for i in range(nb_levels)}
all_labels = []
all_modalities = []
all_subjects = []
all_content_indices = {}  # dict mapping level -> view-0 (T1) content indices
all_content_indices_v1 = {}  # dict mapping level -> view-1 (T2) content indices (only set for per-view masks)

print(f"Extracting latent features from {len(items)} subjects (T1 + T2 each) ...")

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# Map modality name → view_idx for separate-encoder models.
# view 0 = T1 (first modality), view 1 = T2 (second modality).
_VIEW_IDX = {"T1": 0, "T2": 1}

# Detect per-view masks: separate encoders with channel_logits_v1
_has_per_view = (
    getattr(vqvae_module, "separate_encoders", False) and getattr(vqvae_module, "channel_logits_v1", None) is not None
)

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            # Get soft_content_masks (7th returned element)
            _, _, enc_features, _, _, _, soft_masks, *_ = vqvae_model(
                img,
                return_recon=False,
                pool_only=True,
                view_idx=_VIEW_IDX.get(modality, 0),
            )

        # enc_features is a list of (1, hidden_channels) pooled tensors per level.
        for level_idx, pooled in enumerate(enc_features):
            all_features[f"level_{level_idx}"].append(pooled.squeeze(0).cpu().float().numpy())

        # Record the content indices per level.
        #  - Tuple masks: training-style n_views=2 with per-view masks (unused
        #    here since we run n_views=1, but handled for completeness).
        #  - Single-tensor mask: single-view forward.  When per-view learned
        #    masks are active the returned tensor is the view-specific mask
        #    (channel_logits_v1 when view_idx=1), so T2 must be stored in
        #    all_content_indices_v1 to preserve the per-view split.
        for lvl, mask in soft_masks.items():
            if isinstance(mask, tuple):
                mask_v0, mask_v1 = mask
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask_v0.bool())[-1].tolist()
                elif modality == "T2" and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask_v1.bool())[-1].tolist()
            else:
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask.bool())[-1].tolist()
                elif modality == "T2" and _has_per_view and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask.bool())[-1].tolist()

        all_labels.append(item["label"])
        all_modalities.append(modality)
        all_subjects.append(item.get("subject", idx))

# Convert to numpy
for key in all_features:
    all_features[key] = np.array(all_features[key])
all_labels = np.array(all_labels)
all_modalities = np.array(all_modalities)

print(f"\nDone. {len(all_labels)} samples total.")
for key, val in all_features.items():
    print(f"  {key}: {val.shape}")

# Derive content / style subsets from the hidden_channels features for each masked level.
all_style_indices = {}
all_style_indices_v1 = {}

for lvl in all_content_indices.keys():
    all_style_indices[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices[lvl])]
    if _has_per_view and lvl in all_content_indices_v1:
        all_style_indices_v1[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices_v1[lvl])]
    else:
        all_style_indices_v1[lvl] = all_style_indices[lvl]

if len(all_content_indices.get(0, [])) > 0:
    for lvl in all_content_indices.keys():
        print(f"\n--- Level {lvl} ---")
        print(f"  content indices v0 ({len(all_content_indices[lvl])} ch): {all_content_indices[lvl][:8]}...")
        print(f"  style indices   v0 ({len(all_style_indices[lvl])} ch):   {all_style_indices[lvl][:8]}...")
        if _has_per_view and lvl in all_content_indices_v1:
            print(f"  content indices v1 ({len(all_content_indices_v1[lvl])} ch): {all_content_indices_v1[lvl][:8]}...")
            print(f"  style indices   v1 ({len(all_style_indices_v1[lvl])} ch):   {all_style_indices_v1[lvl][:8]}...")

## 4. Feature statistics

In [ ]:
print("=" * 65)
print("Feature Statistics by Level (pooled, from hidden_channels space)")
print("=" * 65)

t1_mask = all_modalities == "T1"
t2_mask = all_modalities == "T2"

for level_idx in range(nb_levels):
    features = all_features[f"level_{level_idx}"]
    t1_f = features[t1_mask]
    t2_f = features[t2_mask]
    paired_dist = np.linalg.norm(t1_f - t2_f, axis=1)
    print(f"\nLevel {level_idx}  (shape {features.shape}):")
    print(
        f"  mean={features.mean():.4f}  std={features.std():.4f}  "
        f"min={features.min():.4f}  max={features.max():.4f}"
    )
    print(f"  T1-T2 paired ℓ2 distance — mean: {paired_dist.mean():.4f}  std: {paired_dist.std():.4f}")

# Extra breakdown for content vs style channels
if len(all_content_indices.get(0, [])) > 0:
    for lvl in all_content_indices.keys():
        lvl_feats = all_features[f"level_{lvl}"]
        print(f"\n--- Level {lvl} channel breakdown ---")
        content_f = lvl_feats[:, all_content_indices[lvl]]
        style_f = lvl_feats[:, all_style_indices[lvl]] if len(all_style_indices[lvl]) > 0 else None
        ct1 = content_f[t1_mask]
        ct2 = content_f[t2_mask]
        print(
            f"  Content  ({len(all_content_indices[lvl])} ch)  — T1-T2 paired dist: {np.linalg.norm(ct1-ct2,axis=1).mean():.4f}"
        )
        if style_f is not None:
            st1 = style_f[t1_mask]
            st2 = style_f[t2_mask]
            print(
                f"  Style    ({len(all_style_indices[lvl])} ch)  — T1-T2 paired dist: {np.linalg.norm(st1-st2,axis=1).mean():.4f}"
            )
        print("  (Content dist should be smaller than style dist if disentanglement is working)")

## 5. PCA visualisation

In [ ]:
colors_by_label = plt.cm.tab10(np.linspace(0, 1, max(len(label_map), 3)))

fig, axes = plt.subplots(2, nb_levels, figsize=(5 * nb_levels, 9))
if nb_levels == 1:
    axes = axes[:, np.newaxis]  # keep 2-D indexing

t1_mask = all_modalities == "T1"
t2_mask = all_modalities == "T2"

for level_idx in range(nb_levels):
    # Only show content dims if a content mask is applied at this level — these are/
    # what the contrastive loss acts on. Plotting all embed_dim mixes in style dims,
    # which encode T1/T2 contrast differences and will dominate the first PC.
    # The full content+style breakdown is in Section 7.
    if level_idx in all_content_indices:
        lvl_feats = all_features[f"level_{level_idx}"]
        # With per-view masks, T1 and T2 have different content channels.
        # Select the correct content channels for each modality, then
        # concatenate back.
        if _has_per_view and level_idx in all_content_indices_v1:
            f_t1 = lvl_feats[t1_mask][:, all_content_indices[level_idx]]
            f_t2 = lvl_feats[t2_mask][:, all_content_indices_v1[level_idx]]
            features = np.empty((len(lvl_feats), f_t1.shape[1]))
            features[t1_mask] = f_t1
            features[t2_mask] = f_t2

            level_label = f"Level {level_idx} — content ({f_t1.shape[1]} dims, per-view mask)"
        else:
            features = lvl_feats[:, all_content_indices[level_idx]]
            level_label = f"Level {level_idx} — content ({len(all_content_indices[level_idx])} dims)"
    else:
        features = all_features[f"level_{level_idx}"]
        level_label = f"Level {level_idx}"

    # L2 normalize features before PCA to prevent scale differences from dominating the first components.
    features = features / np.linalg.norm(features, axis=1, keepdims=True)
    pca = PCA(n_components=2)
    f2d = pca.fit_transform(features)
    ev = pca.explained_variance_ratio_

    # Row 0 — colour by diagnosis
    ax = axes[0, level_idx]
    for lab_idx, lab_name in label_names.items():
        m = all_labels == lab_idx
        ax.scatter(
            f2d[m, 0],
            f2d[m, 1],
            c=[colors_by_label[lab_idx]],
            label=lab_name,
            alpha=0.65,
            s=18,
        )
    ax.set_title(f"{level_label} — Diagnosis\n({ev.sum()*100:.1f}% var)")
    ax.legend(fontsize=8)
    ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

    # Row 1 — colour by modality
    ax = axes[1, level_idx]
    for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
        m = all_modalities == mod
        ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
    ax.set_title(f"{level_label} — Modality (T1 vs T2)")
    ax.legend(fontsize=8)
    ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

plt.suptitle(
    "VQ-VAE-2 Encoder Features — PCA\n" "(Masked levels: content dims only; other levels pooled from hidden space)",
    y=1.02,
    fontsize=12,
)
plt.tight_layout()
plt.savefig("latent_features_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved latent_features_pca.png")

## 6. t-SNE visualisation

In [ ]:
fig, axes = plt.subplots(2, nb_levels, figsize=(5 * nb_levels, 9))
if nb_levels == 1:
    axes = axes[:, np.newaxis]

for level_idx in range(nb_levels):
    # Apply the same content-only filtering as the PCA section for masked levels,
    # including per-view mask handling so T2 uses its own content indices.
    if level_idx in all_content_indices:
        lvl_feats = all_features[f"level_{level_idx}"]
        if _has_per_view and level_idx in all_content_indices_v1:
            f_t1 = lvl_feats[t1_mask][:, all_content_indices[level_idx]]
            f_t2 = lvl_feats[t2_mask][:, all_content_indices_v1[level_idx]]
            features = np.empty((len(lvl_feats), f_t1.shape[1]))
            features[t1_mask] = f_t1
            features[t2_mask] = f_t2
            level_label = f"Level {level_idx} — content ({f_t1.shape[1]} dims, per-view mask)"
        else:
            features = lvl_feats[:, all_content_indices[level_idx]]
            level_label = f"Level {level_idx} — content ({len(all_content_indices[level_idx])} dims)"
    else:
        features = all_features[f"level_{level_idx}"]
        level_label = f"Level {level_idx}"

    # L2 normalize features
    features = features / np.linalg.norm(features, axis=1, keepdims=True)
    tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=42)
    f2d = tsne.fit_transform(features)

    # Row 0 — colour by diagnosis
    ax = axes[0, level_idx]
    for lab_idx, lab_name in label_names.items():
        m = all_labels == lab_idx
        ax.scatter(
            f2d[m, 0],
            f2d[m, 1],
            c=[colors_by_label[lab_idx]],
            label=lab_name,
            alpha=0.65,
            s=18,
        )
    ax.set_title(f"{level_label} — Diagnosis")
    ax.legend(fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

    # Row 1 — colour by modality
    ax = axes[1, level_idx]
    for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
        m = all_modalities == mod
        ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
    ax.set_title(f"{level_label} — Modality (T1 vs T2)")
    ax.legend(fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(
    "VQ-VAE-2 Encoder Features — t-SNE\n" "(Masked levels: content dims only; other levels pooled from hidden space)",
    y=1.02,
    fontsize=12,
)
plt.tight_layout()
plt.savefig("latent_features_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved latent_features_tsne.png")

## 7. Content vs Style channel visualisation (level 0)

The Gumbel mask at level 0 selects `content_channels` out of `embed_dim` dimensions.
If the model has learned to disentangle content (shared across T1/T2) from style
(view-specific), the **content** PCA should show T1 and T2 mixed together, while
the **style** PCA should separate them clearly.

In [ ]:
if 0 in all_content_indices and len(all_style_indices.get(0, [])) > 0:
    level0_feats = all_features["level_0"]
    content_feats = level0_feats[:, all_content_indices[0]]
    style_feats = level0_feats[:, all_style_indices[0]]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    for row, (feats, title_sfx) in enumerate(
        [
            (content_feats, f"Content ({len(all_content_indices.get(0, []))} ch)"),
            (style_feats, f"Style   ({len(all_style_indices.get(0, []))} ch)"),
        ]
    ):
        pca = PCA(n_components=2)
        f2d = pca.fit_transform(feats)
        ev = pca.explained_variance_ratio_

        # Col 0 — by diagnosis
        ax = axes[row, 0]
        for lab_idx, lab_name in label_names.items():
            m = all_labels == lab_idx
            ax.scatter(f2d[m, 0], f2d[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.65, s=18)
        ax.set_title(f"{title_sfx} — Diagnosis\n({ev.sum()*100:.1f}% var)")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

        # Col 1 — by modality
        ax = axes[row, 1]
        for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
            m = all_modalities == mod
            ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
        ax.set_title(f"{title_sfx} — Modality")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

    plt.suptitle("Level 0 — Content vs Style (encoder hidden_channels)", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig("content_vs_style_pca.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Saved content_vs_style_pca.png")
else:
    print("Skipped — channel_logits not active or all channels are content.")

In [ ]:
# ── Spatial code maps: content vs style at level 0 ───────────────────────────
N_EXAMPLES = 3  # number of subjects to visualise
has_style_cb = len(vqvae_module.style_codebooks) > 0

print(f"Style codebooks active: {has_style_cb}")
if 0 in all_content_indices:
    print(f"Content channels (level 0): {len(all_content_indices.get(0, []))}/{hidden_channels}")
    print(f"Style channels   (level 0): {len(all_style_indices.get(0, []))}/{hidden_channels}")

# Pick example subjects (one per diagnosis class if possible)
example_indices = []
seen_labels = set()
for i, item in enumerate(items):
    if item["label"] not in seen_labels:
        example_indices.append(i)
        seen_labels.add(item["label"])
    if len(example_indices) >= N_EXAMPLES:
        break
# Fill remaining slots if fewer classes than N_EXAMPLES
for i in range(len(items)):
    if len(example_indices) >= N_EXAMPLES:
        break
    if i not in example_indices:
        example_indices.append(i)

print(f"\nExtracting spatial code maps for {len(example_indices)} subjects ...")

for ex_idx in example_indices:
    item = items[ex_idx]
    subj = item.get("subject", ex_idx)
    label = label_names[item["label"]]

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            recon, diffs, _, _, _, id_outputs_raw, _, style_id_out = vqvae_model(
                img,
                return_recon=True,
                pool_only=False,
                view_idx=0,
            )

        # id_outputs_raw is coarsest-first; reverse to finest-first
        id_outputs_ex = id_outputs_raw[::-1]
        content_codes = id_outputs_ex[0].squeeze(0).cpu().numpy()  # (D, H, W)

        style_codes = None
        if has_style_cb and 0 in style_id_out:
            style_codes = style_id_out[0].squeeze(0).cpu().numpy()

        # Determine number of columns: 3 slices x (content + style if available)
        n_maps = 2 if style_codes is not None else 1
        fig, axes = plt.subplots(n_maps, 3, figsize=(15, 5 * n_maps), squeeze=False)
        fig.suptitle(
            f"Subject {subj} ({label}, {modality}) — Level 0 Spatial Code Maps\n"
            f"Code map shape: {content_codes.shape}",
            fontsize=13,
            fontweight="bold",
        )

        # Slice indices (middle of each axis)
        mid_d, mid_h, mid_w = [s // 2 for s in content_codes.shape]
        slices_content = [
            ("Axial (d={})".format(mid_d), content_codes[mid_d, :, :]),
            ("Coronal (h={})".format(mid_h), content_codes[:, mid_h, :]),
            ("Sagittal (w={})".format(mid_w), content_codes[:, :, mid_w]),
        ]

        for col, (slice_name, slice_data) in enumerate(slices_content):
            ax = axes[0, col]
            im = ax.imshow(slice_data, cmap="tab20", interpolation="nearest", aspect="auto")
            ax.set_title(f"Content — {slice_name}")
            ax.set_xlabel("dim 1")
            ax.set_ylabel("dim 0")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        if style_codes is not None:
            slices_style = [
                ("Axial (d={})".format(mid_d), style_codes[mid_d, :, :]),
                ("Coronal (h={})".format(mid_h), style_codes[:, mid_h, :]),
                ("Sagittal (w={})".format(mid_w), style_codes[:, :, mid_w]),
            ]
            for col, (slice_name, slice_data) in enumerate(slices_style):
                ax = axes[1, col]
                im = ax.imshow(slice_data, cmap="tab20", interpolation="nearest", aspect="auto")
                ax.set_title(f"Style — {slice_name}")
                ax.set_xlabel("dim 1")
                ax.set_ylabel("dim 0")
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plt.tight_layout()
        plt.show()

        # ── Per-entry frequency comparison: content vs style codebook ────────
        if style_codes is not None:
            fig, axes = plt.subplots(1, 2, figsize=(16, 4))
            fig.suptitle(f"Subject {subj} ({label}) — Codebook Entry Frequency (Level 0)", fontsize=12)

            content_nb = vqvae_module.codebooks[0].n_embed
            style_nb = vqvae_module.style_codebooks["0"].n_embed

            c_hist = np.bincount(content_codes.ravel(), minlength=content_nb).astype(float)
            c_hist /= c_hist.sum()
            axes[0].bar(range(content_nb), c_hist, color="steelblue", alpha=0.7)
            axes[0].set_title(f"Content codebook ({content_nb} entries, {len(all_content_indices.get(0, []))} ch)")
            axes[0].set_xlabel("Entry index")
            axes[0].set_ylabel("Frequency")

            s_hist = np.bincount(style_codes.ravel(), minlength=style_nb).astype(float)
            s_hist /= s_hist.sum()
            axes[1].bar(range(style_nb), s_hist, color="tomato", alpha=0.7)
            axes[1].set_title(f"Style codebook ({style_nb} entries, {len(all_style_indices.get(0, []))} ch)")
            axes[1].set_xlabel("Entry index")
            axes[1].set_ylabel("Frequency")

            plt.tight_layout()
            plt.show()

            used_c = (c_hist > 0).sum()
            used_s = (s_hist > 0).sum()
            print(f"  {subj}: content uses {used_c}/{content_nb} entries, " f"style uses {used_s}/{style_nb} entries")
        else:
            # No style codebook — just show content entry histogram
            content_nb = vqvae_module.codebooks[0].n_embed
            c_hist = np.bincount(content_codes.ravel(), minlength=content_nb).astype(float)
            c_hist /= c_hist.sum()
            used_c = (c_hist > 0).sum()

            fig, ax = plt.subplots(figsize=(10, 3))
            ax.bar(range(content_nb), c_hist, color="steelblue", alpha=0.7)
            ax.set_title(
                f"Subject {subj} ({label}) — Content Codebook Entry Frequency "
                f"({used_c}/{content_nb} used, {len(all_content_indices.get(0, []))} content ch)"
            )
            ax.set_xlabel("Entry index")
            ax.set_ylabel("Frequency")
            plt.tight_layout()
            plt.show()

    del recon, diffs, id_outputs_raw, style_id_out

print("\n✓ Spatial code map visualisation complete.")

## 7a. Spatial Activation Maps — Content vs Style Channels (Level 0)

For a few example subjects, extract the **full spatial encoder output** at level 0 and split it
into content and style channels. Visualise the mean activation (and top individual channels)
across axial, coronal, and sagittal mid-slices to see what spatial patterns each subspace captures.

In [ ]:
# ── Spatial activation maps: content vs style channels at level 0 ────────────
N_EXAMPLES = 3  # one per diagnosis class

# Pick one subject per class
example_indices = []
seen_labels = set()
for i, item in enumerate(items):
    if item["label"] not in seen_labels:
        example_indices.append(i)
        seen_labels.add(item["label"])
    if len(example_indices) >= N_EXAMPLES:
        break

content_idx = np.array(all_content_indices[0]) if 0 in all_content_indices else np.arange(hidden_channels)
style_idx = np.array(all_style_indices.get(0, [])) if len(all_style_indices.get(0, [])) > 0 else None

print(f"Content channels: {len(content_idx)}, Style channels: {len(style_idx) if style_idx is not None else 0}")
print(f"Visualising {len(example_indices)} subjects ...\n")

# Run encoder only (no codebook/decoder) to get spatial feature maps
enc_stack = vqvae_module.encoders
if vqvae_module.separate_encoders and vqvae_module.encoders_v1 is not None:
    enc_stack_v1 = vqvae_module.encoders_v1
else:
    enc_stack_v1 = enc_stack

TOP_K_CHANNELS = 4  # number of individual channels to show

for ex_idx in example_indices:
    item = items[ex_idx]
    subj = item.get("subject", ex_idx)
    label = label_names[item["label"]]

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)
    img = transformed["image_t1"].unsqueeze(0).to(DEVICE)

    # Forward through encoder stack, collect output at each level
    enc_input = img
    spatial_outputs = []
    with torch.no_grad():
        for enc in enc_stack:
            enc_input = enc(enc_input)
            spatial_outputs.append(enc_input)

    # Level 0 spatial features: (1, hidden_channels, D, H, W)
    feat_map = spatial_outputs[0].squeeze(0).cpu().numpy()  # (C, D, H, W)

    content_map = feat_map[content_idx]  # (n_content, D, H, W)
    style_map = feat_map[style_idx] if style_idx is not None and len(style_idx) > 0 else None

    # Mid-slice indices
    _, D, H, W = feat_map.shape
    mid_d, mid_h, mid_w = D // 2, H // 2, W // 2

    # ── Mean activation maps ────────────────────────────────────────────────
    n_rows = 2 if style_map is not None else 1
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows), squeeze=False)
    fig.suptitle(
        f"Subject {subj} ({label}) — Level 0 Mean Activation Maps\n"
        f"Feature map: {feat_map.shape[0]} ch × {D}×{H}×{W}",
        fontsize=13,
        fontweight="bold",
    )

    mean_content = content_map.mean(axis=0)  # (D, H, W)
    for col, (name, slc) in enumerate(
        [
            (f"Axial (d={mid_d})", mean_content[mid_d]),
            (f"Coronal (h={mid_h})", mean_content[:, mid_h]),
            (f"Sagittal (w={mid_w})", mean_content[:, :, mid_w]),
        ]
    ):
        ax = axes[0, col]
        im = ax.imshow(slc, cmap="inferno", interpolation="bilinear", aspect="auto")
        ax.set_title(f"Content mean ({len(content_idx)} ch) — {name}")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    if style_map is not None:
        mean_style = style_map.mean(axis=0)
        for col, (name, slc) in enumerate(
            [
                (f"Axial (d={mid_d})", mean_style[mid_d]),
                (f"Coronal (h={mid_h})", mean_style[:, mid_h]),
                (f"Sagittal (w={mid_w})", mean_style[:, :, mid_w]),
            ]
        ):
            ax = axes[1, col]
            im = ax.imshow(slc, cmap="inferno", interpolation="bilinear", aspect="auto")
            ax.set_title(f"Style mean ({len(style_idx)} ch) — {name}")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

    # ── Top individual channels by variance (most informative) ──────────────
    for subset_name, subset_map, idx_arr in [
        ("Content", content_map, content_idx),
        ("Style", style_map, style_idx),
    ]:
        if subset_map is None:
            continue
        # Rank channels by spatial variance
        ch_var = subset_map.reshape(subset_map.shape[0], -1).var(axis=1)
        top_ch = np.argsort(ch_var)[::-1][:TOP_K_CHANNELS]

        fig, axes = plt.subplots(TOP_K_CHANNELS, 3, figsize=(14, 3.5 * TOP_K_CHANNELS), squeeze=False)
        fig.suptitle(
            f"Subject {subj} ({label}) — Top-{TOP_K_CHANNELS} {subset_name} Channels by Variance (Level 0)",
            fontsize=13,
            fontweight="bold",
        )

        for row, ch_local in enumerate(top_ch):
            ch_global = idx_arr[ch_local]
            ch_data = subset_map[ch_local]  # (D, H, W)
            for col, (name, slc) in enumerate(
                [
                    (f"Axial (d={mid_d})", ch_data[mid_d]),
                    (f"Coronal (h={mid_h})", ch_data[:, mid_h]),
                    (f"Sagittal (w={mid_w})", ch_data[:, :, mid_w]),
                ]
            ):
                ax = axes[row, col]
                im = ax.imshow(slc, cmap="inferno", interpolation="bilinear", aspect="auto")
                if col == 0:
                    ax.set_ylabel(f"Ch {ch_global}\n(var={ch_var[ch_local]:.4f})", fontsize=9)
                if row == 0:
                    ax.set_title(name)
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plt.tight_layout()
        plt.show()

    del spatial_outputs, feat_map

print("✓ Spatial activation map visualisation complete.")

## 7a-ii. Content vs Style Overlay — Colocalization Map

Overlay content and style activation maps on the same MRI slice so we can directly compare
*where* each subspace is active. Uses **L2-norm across channels** as the saliency per voxel
(standard activation-map reduction), upsampled to input resolution and shown as:

- **Red channel** = content activation (normalized per-map)
- **Blue channel** = style activation (normalized per-map)
- **Purple** = both active, **pure red/blue** = one stream dominates

Shown per-subject (one per class) and as a **batch-mean** saliency across all extracted
subjects to smooth out per-subject noise.


In [ ]:
# ── Content vs Style overlay: per-subject + batch-mean saliency ───────────────
import torch.nn.functional as F

content_idx_t = torch.as_tensor(all_content_indices[0], dtype=torch.long) if 0 in all_content_indices else None
style_idx_t = (
    torch.as_tensor(all_style_indices.get(0, []), dtype=torch.long) if len(all_style_indices.get(0, [])) > 0 else None
)

assert (
    content_idx_t is not None and style_idx_t is not None and len(style_idx_t) > 0
), "Need both content and style channels at level 0 for overlay."

enc_stack = vqvae_module.encoders


def _spatial_saliency(feat_map, idx):
    """L2 norm across the given channel subset -> (D, H, W) saliency."""
    sub = feat_map.index_select(0, idx.to(feat_map.device))  # (n, D, H, W)
    return sub.pow(2).sum(dim=0).sqrt()  # (D, H, W)


def _upsample_to(vol, target_shape):
    """vol: (D,H,W) tensor -> (D',H',W') nearest-to-target via trilinear."""
    v = vol[None, None].float()
    up = F.interpolate(v, size=target_shape, mode="trilinear", align_corners=False)
    return up[0, 0]


def _norm01(x, eps=1e-8):
    x = x - x.min()
    m = x.max()
    return x / (m + eps)


def _rgb_overlay(bg, content_sal, style_sal, alpha=0.55):
    """bg, content_sal, style_sal: 2D arrays same shape. Returns (H,W,3) RGB image."""
    bg = _norm01(bg)
    c = _norm01(content_sal)
    s = _norm01(style_sal)
    base = np.stack([bg, bg, bg], axis=-1)
    overlay = np.stack([c, np.zeros_like(c), s], axis=-1)  # R=content, B=style
    return (1 - alpha) * base + alpha * overlay


# Pick one subject per class (reuse logic from previous cell)
example_indices = []
seen_labels = set()
for i, item in enumerate(items):
    if item["label"] not in seen_labels:
        example_indices.append(i)
        seen_labels.add(item["label"])
    if len(example_indices) >= 3:
        break

# Accumulators for batch-mean (across ALL items, both modalities)
sum_content_sal = None
sum_style_sal = None
n_acc = 0
per_subject_cache = {}  # ex_idx -> (img_np, content_sal_np, style_sal_np)

print(f"Computing saliency maps across {len(items)} samples ...")
for i, item in enumerate(items):
    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)
    img = transformed["image_t1"].unsqueeze(0).to(DEVICE)

    enc_input = img
    with torch.no_grad():
        for enc in enc_stack:
            enc_input = enc(enc_input)
            break  # only need level 0 output
        feat = enc_input.squeeze(0)  # (C, D, H, W)

        content_sal = _spatial_saliency(feat, content_idx_t)
        style_sal = _spatial_saliency(feat, style_idx_t)

        # Upsample to input resolution
        target = img.shape[-3:]
        content_sal_up = _upsample_to(content_sal, target).cpu().numpy()
        style_sal_up = _upsample_to(style_sal, target).cpu().numpy()

    # Per-sample max-normalize before accumulating so subjects with larger
    # activation magnitudes do not dominate the batch mean (each subject contributes
    # equally to the "where does the model typically look" spatial pattern).
    content_sal_n = content_sal_up / (content_sal_up.max() + 1e-8)
    style_sal_n = style_sal_up / (style_sal_up.max() + 1e-8)

    if sum_content_sal is None:
        sum_content_sal = np.zeros_like(content_sal_n)
        sum_style_sal = np.zeros_like(style_sal_n)
    sum_content_sal += content_sal_n
    sum_style_sal += style_sal_n
    n_acc += 1

    if i in example_indices:
        per_subject_cache[i] = (img.squeeze().cpu().numpy(), content_sal_up, style_sal_up)

mean_content_sal = sum_content_sal / n_acc
mean_style_sal = sum_style_sal / n_acc
print(f"Accumulated {n_acc} samples.\n")


# ── Plot per-subject overlays + batch mean ───────────────────────────────────
def _plot_overlay_row(axes_row, img_np, c_sal, s_sal, title_prefix):
    D, H, W = img_np.shape
    md, mh, mw = D // 2, H // 2, W // 2
    views = [
        ("Axial", img_np[md], c_sal[md], s_sal[md]),
        ("Coronal", img_np[:, mh], c_sal[:, mh], s_sal[:, mh]),
        ("Sagittal", img_np[:, :, mw], c_sal[:, :, mw], s_sal[:, :, mw]),
    ]
    for col, (name, bg, c, s) in enumerate(views):
        ax = axes_row[col]
        ax.imshow(_rgb_overlay(bg, c, s), interpolation="bilinear", aspect="auto")
        ax.set_title(f"{title_prefix} — {name}")
        ax.set_xticks([])
        ax.set_yticks([])


n_rows = len(per_subject_cache) + 1  # +1 for batch mean (uses subject-0 bg)
fig, axes = plt.subplots(n_rows, 3, figsize=(14, 4.2 * n_rows), squeeze=False)
fig.suptitle(
    "Content (red) vs Style (blue) Activation Overlay — Level 0\n"
    "Purple = both active, red = content-only, blue = style-only",
    fontsize=13,
    fontweight="bold",
)

for row, ex_idx in enumerate(per_subject_cache):
    img_np, c_sal, s_sal = per_subject_cache[ex_idx]
    label = label_names[items[ex_idx]["label"]]
    subj = items[ex_idx].get("subject", ex_idx)
    _plot_overlay_row(axes[row], img_np, c_sal, s_sal, f"{subj} ({label})")

# Batch mean row — use the first cached subject's MRI as background for anatomy reference
bg_img = next(iter(per_subject_cache.values()))[0]
_plot_overlay_row(axes[-1], bg_img, mean_content_sal, mean_style_sal, f"Batch mean (n={n_acc})")

# Legend
fig.text(
    0.5,
    -0.01,
    "Red channel = content L2-norm saliency  |  Blue channel = style L2-norm saliency  "
    "(per-sample max-normalized before averaging; overlay re-normalized to [0,1] per map)",
    ha="center",
    fontsize=9,
    style="italic",
)

plt.tight_layout()
plt.show()


# ── Side-by-side: pure content vs pure style for the batch mean ──────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8), squeeze=False)
fig.suptitle(f"Batch-Mean Saliency — Content vs Style (n={n_acc})", fontsize=13, fontweight="bold")

D, H, W = mean_content_sal.shape
md, mh, mw = D // 2, H // 2, W // 2
slices = [("Axial", md, 0), ("Coronal", mh, 1), ("Sagittal", mw, 2)]

for col, (name, mid, axis) in enumerate(slices):
    c_slice = np.take(mean_content_sal, mid, axis=axis)
    s_slice = np.take(mean_style_sal, mid, axis=axis)
    im0 = axes[0, col].imshow(c_slice, cmap="Reds", interpolation="bilinear", aspect="auto")
    axes[0, col].set_title(f"Content mean — {name}")
    plt.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)
    im1 = axes[1, col].imshow(s_slice, cmap="Blues", interpolation="bilinear", aspect="auto")
    axes[1, col].set_title(f"Style mean — {name}")
    plt.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("✓ Overlay + batch-mean saliency visualisation complete.")

## 7b. Per-level modality separation: content vs style

Quantitative check: for each level, split features into content/style channels using
the Gumbel mask, then measure how strongly modality (T1/T2) clusters in each subspace.

- **Silhouette score** close to 1 → strong modality clustering (bad for content, good for style)
- **Silhouette score** close to 0 → modalities are mixed (good for content)

If the contrastive loss is working at a given level, the content silhouette should be
lower than the style silhouette at that level.

In [ ]:
from sklearn.metrics import silhouette_score

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# Gather per-level content/style indices from the model's channel_logits or fixed masks
modality_binary = (all_modalities == "T2").astype(int)

# Check for per-view masks
_has_v1_logits = hasattr(vqvae_module, "channel_logits_v1") and vqvae_module.channel_logits_v1 is not None
_is_fixed = getattr(vqvae_module, "mask_mode", "onthefly") == "fixed"

results = []

for lvl in range(nb_levels):
    feats = all_features[f"level_{lvl}"]
    n_ch = feats.shape[1]

    # Determine content/style indices for this level.
    # Three paths: fixed mask, learned/onthefly with channel_logits, or no mask.
    _has_logits_for_lvl = (
        hasattr(vqvae_module, "channel_logits")
        and len(vqvae_module.channel_logits) > 0
        and str(lvl) in vqvae_module.channel_logits
    )
    _has_fixed_for_lvl = _is_fixed and hasattr(vqvae_module, f"fixed_mask_{lvl}")

    if _has_fixed_for_lvl:
        # Fixed mode: first k channels = content, rest = style
        k = vqvae_module.content_channels_per_level.get(lvl, vqvae_module.content_channels)
        c_idx_v0 = np.arange(k)
        s_idx_v0 = np.arange(k, n_ch)
        c_idx_v1 = c_idx_v0  # fixed mode is always shared across views
        s_idx_v1 = s_idx_v0

        content_feats = feats[:, c_idx_v0]
        style_feats_v0 = feats[:, s_idx_v0] if len(s_idx_v0) > 0 else None
        style_feats_v1 = None

        sil_content = silhouette_score(content_feats, modality_binary) if content_feats.shape[1] >= 2 else float("nan")
        if style_feats_v0 is not None:
            sil_style = (
                silhouette_score(style_feats_v0, modality_binary) if style_feats_v0.shape[1] >= 2 else float("nan")
            )
        else:
            sil_style = float("nan")
        sil_all = silhouette_score(feats, modality_binary)

        results.append(
            {
                "level": lvl,
                "content_ch": k,
                "style_ch": n_ch - k,
                "sil_content": sil_content,
                "sil_style": sil_style,
                "sil_all": sil_all,
            }
        )
        print(
            f"Level {lvl} (fixed):  content sil={sil_content:+.3f} ({k} ch)  |  "
            f"style sil={sil_style:+.3f} ({n_ch - k} ch)  |  "
            f"all sil={sil_all:+.3f}"
        )

    elif _has_logits_for_lvl:
        # Learned / onthefly: use channel_logits to find top-k
        logits_v0 = vqvae_module.channel_logits[str(lvl)].detach().cpu()
        k = vqvae_module.content_channels_per_level.get(lvl, vqvae_module.content_channels)
        c_idx_v0 = logits_v0.topk(k).indices.sort().values.numpy()
        s_idx_v0 = np.array([i for i in range(n_ch) if i not in set(c_idx_v0)])

        if _has_v1_logits and str(lvl) in vqvae_module.channel_logits_v1:
            logits_v1 = vqvae_module.channel_logits_v1[str(lvl)].detach().cpu()
            c_idx_v1 = logits_v1.topk(k).indices.sort().values.numpy()
            s_idx_v1 = np.array([i for i in range(n_ch) if i not in set(c_idx_v1)])

            # Per-view: select correct content/style for each modality
            t1_m = all_modalities == "T1"
            t2_m = all_modalities == "T2"
            content_feats = np.empty((len(feats), k))
            content_feats[t1_m] = feats[t1_m][:, c_idx_v0]
            content_feats[t2_m] = feats[t2_m][:, c_idx_v1]
            # Style dims may differ in count; use all dims for 'all' silhouette
            style_feats_v0 = feats[t1_m][:, s_idx_v0] if len(s_idx_v0) > 0 else None
            style_feats_v1 = feats[t2_m][:, s_idx_v1] if len(s_idx_v1) > 0 else None
        else:
            # Shared mask
            content_feats = feats[:, c_idx_v0]
            style_feats_v0 = feats[:, s_idx_v0] if len(s_idx_v0) > 0 else None
            style_feats_v1 = None
            c_idx_v1 = c_idx_v0
            s_idx_v1 = s_idx_v0

        sil_content = silhouette_score(content_feats, modality_binary) if content_feats.shape[1] >= 2 else float("nan")

        # For style silhouette with per-view masks, use v0 indices for all
        # (style dims may differ; this is an approximation)
        if style_feats_v0 is not None and not _has_v1_logits:
            style_feats_all = feats[:, s_idx_v0]
            sil_style = (
                silhouette_score(style_feats_all, modality_binary) if style_feats_all.shape[1] >= 2 else float("nan")
            )
        else:
            sil_style = float("nan")  # per-view style dims differ, skip

        sil_all = silhouette_score(feats, modality_binary)

        results.append(
            {
                "level": lvl,
                "content_ch": k,
                "style_ch": n_ch - k,
                "sil_content": sil_content,
                "sil_style": sil_style,
                "sil_all": sil_all,
            }
        )
        per_view_note = " (per-view)" if _has_v1_logits else ""
        print(
            f"Level {lvl}{per_view_note}:  content sil={sil_content:+.3f} ({k} ch)  |  "
            f"style sil={sil_style:+.3f} ({n_ch - k} ch)  |  "
            f"all sil={sil_all:+.3f}"
        )
    else:
        sil_all = silhouette_score(feats, modality_binary)
        results.append(
            {
                "level": lvl,
                "content_ch": n_ch,
                "style_ch": 0,
                "sil_content": sil_all,
                "sil_style": float("nan"),
                "sil_all": sil_all,
            }
        )
        print(f"Level {lvl}:  no mask  |  all sil={sil_all:+.3f} ({n_ch} ch)")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results))
w = 0.25
bars_c = ax.bar(x - w, [r["sil_content"] for r in results], w, label="Content", color="steelblue")
bars_s = ax.bar(x, [r["sil_style"] for r in results], w, label="Style", color="tomato")
bars_a = ax.bar(x + w, [r["sil_all"] for r in results], w, label="All", color="gray", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f"Level {r['level']}" for r in results])
ax.set_ylabel("Silhouette score (modality)")
ax.set_title("Modality separation by content/style subspace per level\n(lower content = better disentanglement)")
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.legend()
plt.tight_layout()
plt.savefig("modality_silhouette_per_level.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved modality_silhouette_per_level.png")

## 7b. Cross-View Content Alignment (Identifiability Test)

**Core test:** If content is identifiable, the content representation of a T1 scan
should match the content representation of the paired T2 scan of the **same brain**.

For each subject we compute:
- **Content cosine similarity** between T1 and T2 (same subject) → should be **high**
- **Style cosine similarity** between T1 and T2 (same subject) → should be **low**
  (style encodes modality-specific contrast, which differs between T1 and T2)

We also compare against **cross-subject** baselines (random T1–T2 pairings) to show
that the high content similarity is subject-specific, not a degenerate constant mapping.

Additionally, we report **Recall@k** for cross-modality retrieval: given a T1 content
vector, can we retrieve the correct T2 from a gallery by nearest-neighbor search?

In [ ]:
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# ── Split features into paired T1/T2 per subject ────────────────
# Data alternates: T1_subj0, T2_subj0, T1_subj1, T2_subj1, ...
level0_feats = all_features["level_0"]
n_subjects = len(level0_feats) // 2
t1_idx = np.arange(0, 2 * n_subjects, 2)
t2_idx = np.arange(1, 2 * n_subjects, 2)

assert np.all(all_modalities[t1_idx] == "T1"), "T1/T2 alternation assumption broken"
assert np.all(all_modalities[t2_idx] == "T2"), "T1/T2 alternation assumption broken"

content_idx = list(all_content_indices[0]) if 0 in all_content_indices else []
content_idx_v1 = list(all_content_indices_v1[0]) if 0 in all_content_indices_v1 else content_idx
style_idx = all_style_indices.get(0, [])
style_idx_v1 = all_style_indices_v1.get(0, []) if _has_per_view else style_idx

print(f"Subjects: {n_subjects}")
print(f"Content dims v0: {len(content_idx)},  Style dims v0: {len(style_idx)}")
if _has_per_view:
    print(f"Content dims v1: {len(content_idx_v1)},  Style dims v1: {len(style_idx_v1)}")
    overlap = set(content_idx) & set(content_idx_v1)
    print(f"Content channel overlap: {len(overlap)} / {len(content_idx)}  (indices shared by both views)")

# ── Extract content and style features ───────────────────────────
# With per-view masks, v0 and v1 select DIFFERENT physical channels as "content".
# Comparing raw indexed channels across views is meaningless — the k-th content
# dimension of v0 has no correspondence to the k-th content dimension of v1.
#
# Instead, we compare in the ALIGNED k-dimensional space: each view's features
# are masked and then the selected channels are extracted, producing k-dim
# vectors that the contrastive loss actually operates on.
t1_content = level0_feats[t1_idx][:, content_idx]  # (N, k)
t2_content = level0_feats[t2_idx][:, content_idx_v1]  # (N, k)
t1_style = level0_feats[t1_idx][:, style_idx]  # (N, style_dim)
t2_style = level0_feats[t2_idx][:, style_idx_v1]

# Per-level content/style indices and paired features
level_content = {}
for lvl in range(nb_levels):
    feats = all_features[f"level_{lvl}"]
    c_idx = list(all_content_indices[lvl]) if lvl in all_content_indices else []
    c_idx_v1 = list(all_content_indices_v1[lvl]) if lvl in all_content_indices_v1 else c_idx
    s_idx = all_style_indices.get(lvl, [])
    s_idx_v1 = all_style_indices_v1.get(lvl, s_idx) if _has_per_view else s_idx
    level_content[lvl] = {
        "t1": feats[t1_idx],
        "t2": feats[t2_idx],
        "content_idx": c_idx,
        "content_idx_v1": c_idx_v1,
        "style_idx": s_idx,
        "style_idx_v1": s_idx_v1,
    }


# ── 1. Same-subject cosine similarity ───────────────────────────
def paired_cosine_sim(A, B):
    """Row-wise cosine similarity between paired rows of A and B."""
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-8)
    return np.sum(A_norm * B_norm, axis=1)


same_content_sim = paired_cosine_sim(t1_content, t2_content)
same_style_sim = paired_cosine_sim(t1_style, t2_style)

# Per-level same-subject similarity (all dims at that level)
same_level_sim = {}
for lvl in range(nb_levels):
    same_level_sim[lvl] = paired_cosine_sim(level_content[lvl]["t1"], level_content[lvl]["t2"])

# ── 2. Cross-subject baseline ───────────────────────────────────
# For per-view masks: cross-subject similarity must also use
# per-view indexed features (t1 with content_idx, t2 with content_idx_v1).
content_sim_matrix = sklearn_cosine(t1_content, t2_content)  # (N, N)
style_sim_matrix = sklearn_cosine(t1_style, t2_style)


# ── 3. L2-normalised comparison (what the MoCo loss actually sees) ──
# The contrastive loss L2-normalises content features before computing
# dot products. Replicate that here for a fair comparison.
def l2_normalize(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.clip(norms, 1e-8, None)


t1_content_norm = l2_normalize(t1_content)
t2_content_norm = l2_normalize(t2_content)

same_content_sim_norm = np.sum(t1_content_norm * t2_content_norm, axis=1)
cross_content_sim_norm = sklearn_cosine(t1_content_norm, t2_content_norm)

# ── 4. Print summary ────────────────────────────────────────────
print(f"\nSame-subject cosine similarity (mean +/- std):")
print(f"  Content: {same_content_sim.mean():.3f} +/- {same_content_sim.std():.3f}")
print(f"  Style:   {same_style_sim.mean():.3f} +/- {same_style_sim.std():.3f}")

print(f"\nSame-subject cosine similarity by level (all dims):")
for lvl in sorted(same_level_sim.keys()):
    s = same_level_sim[lvl]
    print(f"  Level {lvl}: {s.mean():.3f} +/- {s.std():.3f}")

# Cross-subject: mean of off-diagonal entries
n = content_sim_matrix.shape[0]
off_diag_mask = ~np.eye(n, dtype=bool)
cross_content_mean = content_sim_matrix[off_diag_mask].mean()
cross_content_std = content_sim_matrix[off_diag_mask].std()
cross_style_mean = style_sim_matrix[off_diag_mask].mean()
cross_style_std = style_sim_matrix[off_diag_mask].std()

print(f"\nCross-subject cosine similarity (mean +/- std):")
print(f"  Content: {cross_content_mean:.3f} +/- {cross_content_std:.3f}")
print(f"  Style:   {cross_style_mean:.3f} +/- {cross_style_std:.3f}")

# Diagonal = same-subject in the cross-subject matrix
diag_content = np.diag(content_sim_matrix)
diag_style = np.diag(style_sim_matrix)
print(f"\nSanity check — diagonal of cross-subject matrix (= same-subject):")
print(f"  Content: {diag_content.mean():.3f} +/- {diag_content.std():.3f}")
print(f"  Style:   {diag_style.mean():.3f} +/- {diag_style.std():.3f}")

gap = diag_content.mean() - cross_content_mean
print(
    f"\nContent same-vs-cross gap: {gap:+.3f}  {'(GOOD: same > cross)' if gap > 0.05 else '(WARNING: no meaningful gap)'}"
)

# L2-normalised (what loss sees)
print(f"\nL2-normalised content similarity (matches MoCo loss):")
print(f"  Same-subject:  {same_content_sim_norm.mean():.3f} +/- {same_content_sim_norm.std():.3f}")
cross_norm_off = cross_content_sim_norm[off_diag_mask]
print(f"  Cross-subject: {cross_norm_off.mean():.3f} +/- {cross_norm_off.std():.3f}")
gap_norm = same_content_sim_norm.mean() - cross_norm_off.mean()
print(f"  Gap: {gap_norm:+.3f}")

if _has_per_view:
    print(f"\nNOTE: Per-view masks are active. v0 content channels: {content_idx[:5]}...")
    print(f"  v1 content channels: {content_idx_v1[:5]}...")
    print(f"  These are DIFFERENT physical channels. Cosine similarity is computed")
    print(f"  in the k-dim aligned space (position-wise, not channel-identity-wise).")
    print(f"  If channels have no natural ordering correspondence, this metric")
    print(f"  may underestimate true alignment. Consider the per-level (all dims)")
    print(f"  similarities above as a more reliable signal.")

## 7b-ii. Cross-Modal Patch Retrieval Accuracy (Spatial)

For each spatial patch in modality A (T1), find the nearest-neighbour patch in modality B (T2)
across the **entire batch** using cosine similarity on the **content channels** of the spatial
feature map. If content is properly aligned, the retrieved patch should come from the
**same subject and same spatial position**.

Reports:
- **Position-accurate Recall@k**: retrieved patch is from the correct subject AND spatial position
- **Subject-accurate Recall@k**: retrieved patch is from the correct subject (any position)
- **Chance level** for comparison

In [ ]:
import torch.nn.functional as F

# ── Configuration ────────────────────────────────────────────────────────────
PATCH_GRID = settings.get("patch_grid", [4, 5, 4])  # (D, H, W) — must match training
TOP_K = [1, 5, 10]

# ── Extract spatial feature maps for all subjects ────────────────────────────
enc_stack_v0 = vqvae_module.encoders
enc_stack_v1 = (
    vqvae_module.encoders_v1
    if vqvae_module.separate_encoders and vqvae_module.encoders_v1 is not None
    else enc_stack_v0
)

content_idx = list(all_content_indices[0]) if 0 in all_content_indices else list(range(hidden_channels))
content_idx_v1 = list(all_content_indices_v1[0]) if 0 in all_content_indices_v1 else content_idx

print(f"Patch grid: {PATCH_GRID}  =>  {np.prod(PATCH_GRID)} patches per volume")
print(f"Content channels v0: {len(content_idx)}, v1: {len(content_idx_v1)}")
print(f"Extracting spatial features for {len(items)} subjects ...")

t1_patches_all = []  # will be (n_subjects, n_patches, n_content_ch)
t2_patches_all = []

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)
        encs = enc_stack_v0 if modality == "T1" else enc_stack_v1

        with torch.no_grad():
            x = img
            for enc in encs:
                x = enc(x)
                break  # level 0 only

        # x: (1, C, D', H', W') — spatial feature map at level 0
        feat = x  # keep on device for adaptive_avg_pool3d

        # Pool to patch grid
        gD, gH, gW = PATCH_GRID
        patches = F.adaptive_avg_pool3d(feat, (gD, gH, gW))  # (1, C, gD, gH, gW)
        patches = patches.squeeze(0)  # (C, gD, gH, gW)

        # Select content channels and reshape to (n_patches, n_content)
        ci = content_idx if modality == "T1" else content_idx_v1
        content_patches = patches[ci]  # (n_content, gD, gH, gW)
        content_patches = content_patches.reshape(len(ci), -1).T  # (n_patches, n_content)
        content_patches = content_patches.cpu().float().numpy()

        if modality == "T1":
            t1_patches_all.append(content_patches)
        else:
            t2_patches_all.append(content_patches)

t1_patches_all = np.array(t1_patches_all)  # (n_subjects, n_patches, n_content)
t2_patches_all = np.array(t2_patches_all)

n_subjects, n_patches, n_content = t1_patches_all.shape
print(f"\nExtracted: {n_subjects} subjects x {n_patches} patches x {n_content} content dims")

# ── Build gallery and queries ────────────────────────────────────────────────
# Query: every T1 patch  (n_subjects * n_patches, n_content)
# Gallery: every T2 patch (n_subjects * n_patches, n_content)
queries = t1_patches_all.reshape(-1, n_content)  # (N*P, C)
gallery = t2_patches_all.reshape(-1, n_content)  # (N*P, C)

# L2-normalise for cosine similarity
q_norm = queries / (np.linalg.norm(queries, axis=1, keepdims=True) + 1e-8)
g_norm = gallery / (np.linalg.norm(gallery, axis=1, keepdims=True) + 1e-8)

# Cosine similarity matrix: (N*P, N*P)
sim_matrix = q_norm @ g_norm.T

# Ground-truth: query i should match gallery i (same subject, same position)
# Subject ID for each patch
query_subject = np.repeat(np.arange(n_subjects), n_patches)
gallery_subject = np.repeat(np.arange(n_subjects), n_patches)
# Position ID for each patch
query_position = np.tile(np.arange(n_patches), n_subjects)
gallery_position = np.tile(np.arange(n_patches), n_subjects)

# ── Recall@k computation ────────────────────────────────────────────────────
# Sort gallery indices by descending similarity for each query
ranked = np.argsort(-sim_matrix, axis=1)

results = {}
for k in TOP_K:
    top_k_indices = ranked[:, :k]

    # Position-accurate: correct subject AND correct position
    top_k_subj = gallery_subject[top_k_indices]
    top_k_pos = gallery_position[top_k_indices]
    pos_correct = ((top_k_subj == query_subject[:, None]) & (top_k_pos == query_position[:, None])).any(axis=1)
    pos_recall = pos_correct.mean()

    # Subject-accurate: correct subject, any position
    subj_correct = (top_k_subj == query_subject[:, None]).any(axis=1)
    subj_recall = subj_correct.mean()

    results[k] = {"position": pos_recall, "subject": subj_recall}

# Chance levels
chance_pos = 1.0 / (n_subjects * n_patches)
chance_subj = n_patches / (n_subjects * n_patches)

# ── Print results ────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Cross-Modal Patch Retrieval (T1 query -> T2 gallery)")
print(f"{'='*60}")
print(f"  Gallery size: {n_subjects} subjects x {n_patches} patches = {n_subjects * n_patches}")
print(f"  Chance (position): {chance_pos:.4f}")
print(f"  Chance (subject):  {chance_subj:.4f}")
print()
for k in TOP_K:
    r = results[k]
    print(f"  Recall@{k:>2d}  position: {r['position']:.4f}   subject: {r['subject']:.4f}")

# ── Per-subject retrieval accuracy breakdown ─────────────────────────────────
print(f"\nPer-subject position Recall@1:")
for s in range(min(n_subjects, 20)):  # show at most 20
    mask = query_subject == s
    s_correct = (gallery_subject[ranked[mask, 0]] == s) & (gallery_position[ranked[mask, 0]] == query_position[mask])
    label = label_names[items[s]["label"]]
    print(f"  Subject {s:3d} ({label:>4s}): {s_correct.mean():.3f}")

# ── Heatmap: mean position-wise retrieval similarity ─────────────────────────
# For each spatial position, average the same-subject cosine similarity
pos_sim = np.zeros(n_patches)
for p in range(n_patches):
    sims = []
    for s in range(n_subjects):
        qi = s * n_patches + p
        gi = s * n_patches + p
        sims.append(sim_matrix[qi, gi])
    pos_sim[p] = np.mean(sims)

gD, gH, gW = PATCH_GRID
pos_sim_vol = pos_sim.reshape(gD, gH, gW)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    "Mean Same-Subject Cosine Similarity by Spatial Position\n" "(T1 content vs T2 content, per patch)",
    fontsize=13,
    fontweight="bold",
)

mid_d, mid_h, mid_w = gD // 2, gH // 2, gW // 2
for ax, (title, slc) in zip(
    axes,
    [
        (f"Axial (d={mid_d})", pos_sim_vol[mid_d]),
        (f"Coronal (h={mid_h})", pos_sim_vol[:, mid_h]),
        (f"Sagittal (w={mid_w})", pos_sim_vol[:, :, mid_w]),
    ],
):
    im = ax.imshow(slc, cmap="RdYlGn", vmin=0, vmax=1, interpolation="nearest", aspect="auto")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(f"\nSpatial similarity range: {pos_sim.min():.3f} — {pos_sim.max():.3f} (mean {pos_sim.mean():.3f})")

## 7b-iii. PCA / t-SNE of Spatial Patches — Colored by Position & Modality

Visualise whether patches from the **same spatial position** across modalities cluster together.

- **Left**: colored by **spatial region** (coarse grid position) — same-position patches from T1 and T2
  should overlap if content is aligned
- **Right**: colored by **modality** (T1 vs T2) — if content is aligned, there should be NO
  visible separation by modality; patches should intermix

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ── Re-use t1_patches_all / t2_patches_all from previous cell ───────────────
# Shape: (n_subjects, n_patches, n_content)
# Subsample subjects if dataset is large (t-SNE is O(n^2))
MAX_SUBJECTS = 50
if n_subjects > MAX_SUBJECTS:
    rng = np.random.RandomState(42)
    sub_idx = rng.choice(n_subjects, MAX_SUBJECTS, replace=False)
    t1_sub = t1_patches_all[sub_idx]
    t2_sub = t2_patches_all[sub_idx]
    n_sub = MAX_SUBJECTS
else:
    t1_sub = t1_patches_all
    t2_sub = t2_patches_all
    n_sub = n_subjects

# Stack T1 and T2 patches: (2 * n_sub * n_patches, n_content)
all_patches = np.concatenate([t1_sub.reshape(-1, n_content), t2_sub.reshape(-1, n_content)], axis=0)

# Labels
n_pts_per_mod = n_sub * n_patches
modality_labels = np.array(["T1"] * n_pts_per_mod + ["T2"] * n_pts_per_mod)
position_labels = np.tile(np.arange(n_patches), n_sub * 2)  # spatial position index
subject_labels = np.concatenate([np.repeat(np.arange(n_sub), n_patches)] * 2)

# ── PCA ──────────────────────────────────────────────────────────────────────
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(all_patches)

# ── t-SNE ────────────────────────────────────────────────────────────────────
print(f"Running t-SNE on {len(all_patches)} patches ...")
tsne = TSNE(
    n_components=2, perplexity=min(30, len(all_patches) // 4), random_state=42, init="pca", learning_rate="auto"
)
tsne_coords = tsne.fit_transform(all_patches)
print("Done.")

# ── Assign colors for spatial position (use a cyclic colormap) ───────────────
# Group positions into coarse regions for visual clarity
gD, gH, gW = PATCH_GRID
# Use depth slice as region label (most interpretable for brain volumes)
pos_depth = position_labels // (gH * gW)  # which depth slice
pos_colors = plt.cm.tab10(pos_depth / max(gD - 1, 1))

mod_colors = np.where(
    modality_labels[:, None] == "T1", [0.2, 0.5, 0.9, 0.5], [0.9, 0.3, 0.2, 0.5]  # blue, semi-transparent
)  # red, semi-transparent

# ── Plot PCA ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f"PCA of Spatial Content Patches ({n_sub} subjects, {n_patches} patches each)\n"
    f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, "
    f"PC2={pca.explained_variance_ratio_[1]:.1%}",
    fontsize=13,
    fontweight="bold",
)

ax = axes[0]
ax.scatter(pca_coords[:, 0], pca_coords[:, 1], c=pos_colors, s=8, alpha=0.5, edgecolors="none")
ax.set_title(f"Colored by depth slice (0..{gD-1})")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
# Legend for depth slices
for d in range(gD):
    ax.scatter([], [], c=[plt.cm.tab10(d / max(gD - 1, 1))], s=40, label=f"depth {d}")
ax.legend(fontsize=8, markerscale=1.5)

ax = axes[1]
t1_mask = modality_labels == "T1"
ax.scatter(pca_coords[t1_mask, 0], pca_coords[t1_mask, 1], c="steelblue", s=8, alpha=0.4, edgecolors="none", label="T1")
ax.scatter(pca_coords[~t1_mask, 0], pca_coords[~t1_mask, 1], c="tomato", s=8, alpha=0.4, edgecolors="none", label="T2")
ax.set_title("Colored by modality")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(fontsize=10, markerscale=3)

plt.tight_layout()
plt.show()

# ── Plot t-SNE ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f"t-SNE of Spatial Content Patches ({n_sub} subjects, {n_patches} patches each)", fontsize=13, fontweight="bold"
)

ax = axes[0]
ax.scatter(tsne_coords[:, 0], tsne_coords[:, 1], c=pos_colors, s=8, alpha=0.5, edgecolors="none")
ax.set_title(f"Colored by depth slice (0..{gD-1})")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
for d in range(gD):
    ax.scatter([], [], c=[plt.cm.tab10(d / max(gD - 1, 1))], s=40, label=f"depth {d}")
ax.legend(fontsize=8, markerscale=1.5)

ax = axes[1]
ax.scatter(
    tsne_coords[t1_mask, 0], tsne_coords[t1_mask, 1], c="steelblue", s=8, alpha=0.4, edgecolors="none", label="T1"
)
ax.scatter(
    tsne_coords[~t1_mask, 0], tsne_coords[~t1_mask, 1], c="tomato", s=8, alpha=0.4, edgecolors="none", label="T2"
)
ax.set_title("Colored by modality")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(fontsize=10, markerscale=3)

plt.tight_layout()
plt.show()

# ── Quantitative: silhouette score for modality separation ───────────────────
from sklearn.metrics import silhouette_score

# Subsample for speed if needed
MAX_SIL = 5000
if len(all_patches) > MAX_SIL:
    rng = np.random.RandomState(0)
    sil_idx = rng.choice(len(all_patches), MAX_SIL, replace=False)
else:
    sil_idx = np.arange(len(all_patches))

sil_mod = silhouette_score(all_patches[sil_idx], modality_labels[sil_idx], metric="cosine")
sil_pos = silhouette_score(all_patches[sil_idx], position_labels[sil_idx], metric="cosine")

print(f"\nSilhouette scores (cosine, higher = more separated):")
print(
    f"  By modality:  {sil_mod:+.3f}  {'(BAD: modalities still separated)' if sil_mod > 0.1 else '(GOOD: modalities intermixed)'}"
)
print(
    f"  By position:  {sil_pos:+.3f}  {'(GOOD: spatial structure preserved)' if sil_pos > 0.05 else '(WARNING: no spatial structure)'}"
)

## 7c. Content–Style Information Leakage Test

If disentanglement is working, **content should not encode modality** and
**style should not encode diagnosis**:

| Feature set | Predict modality (T1/T2) | Predict diagnosis (AD/CN/MCI) |
|---|---|---|
| Content | ~50% (chance) | **High** (anatomy encodes disease) |
| Style | **High** (contrast is modality-specific) | ~33% (chance) |

We train 5-fold cross-validated logistic regression classifiers on each feature
subset and report accuracy. Deviations from the expected pattern indicate
information leakage between content and style.

We also compute the **alignment** and **uniformity** metrics from
[Wang & Isola 2020] on the content representations, which are more principled
than visual inspection of t-SNE plots:
- **Alignment** (↓ better): expected distance between positive pairs (T1↔T2 same subject)
- **Uniformity** (↓ better): log-average of pairwise Gaussian potential (spread on hypersphere)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, Normalizer, StandardScaler

# ══════════════════════════════════════════════════════════════════
# 1. INFORMATION LEAKAGE TEST — linear (LR) + non-linear (MLP)
# Runs both probes on every per-level feature set (content / style /
# all dims) for modality AND diagnosis.
# ══════════════════════════════════════════════════════════════════

# Prepare labels
modality_labels = (all_modalities == "T2").astype(int)  # 0=T1, 1=T2
diagnosis_labels = all_labels  # integer class labels

n_classes = len(np.unique(diagnosis_labels))
chance_modality = 50.0  # binary
chance_diagnosis = 100.0 / n_classes

# Row masks for per-view indexing (T1 → v0 mask, T2 → v1 mask).
_t1_rows = all_modalities == "T1"
_t2_rows = all_modalities == "T2"


def _per_view_select(feats, idx_v0, idx_v1):
    """Apply v0 indices to T1 rows and v1 indices to T2 rows.

    Matches the training-time probe (eval/cross_reconstruction.py), which
    indexes each view's features with that view's Gumbel content mask.
    Assumes len(idx_v0) == len(idx_v1).
    """
    if list(idx_v0) == list(idx_v1):
        return feats[:, idx_v0]
    out = np.empty((feats.shape[0], len(idx_v0)), dtype=feats.dtype)
    out[_t1_rows] = feats[_t1_rows][:, idx_v0]
    out[_t2_rows] = feats[_t2_rows][:, idx_v1]
    return out


# Feature sets to test — all from embed_dim space
level0_feats = all_features["level_0"]
feature_sets = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    if lc["content_idx"]:
        feature_sets[f"Content (L{lvl})"] = _per_view_select(feats, lc["content_idx"], lc["content_idx_v1"])
    if lc["style_idx"]:
        feature_sets[f"Style (L{lvl})"] = _per_view_select(feats, lc["style_idx"], lc["style_idx_v1"])
    feature_sets[f"All dims (L{lvl})"] = feats


def _lr_probe(multiclass=False):
    kw = dict(max_iter=1000, random_state=42, C=1.0)
    if multiclass:
        kw["multi_class"] = "multinomial"
    return make_pipeline(Normalizer(norm="l2"), StandardScaler(), LogisticRegression(**kw))


def _mlp_probe():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            max_iter=500,
            early_stopping=True,
            random_state=42,
        ),
    )


print("=" * 96)
print("INFORMATION LEAKAGE TEST — Linear (LR) + Non-linear (MLP), 5-fold CV")
print("=" * 96)
print(
    f"\nChance levels: modality={chance_modality:.0f}%, " f"diagnosis={chance_diagnosis:.1f}% ({n_classes} classes)\n"
)

header = (
    f"{'Feature set':<22} "
    f"{'Mod LR':>10} {'Mod MLP':>10} "
    f"{'Diag LR':>10} {'Diag MLP':>10} "
    f"{'Mod leak?':>12} {'Diag leak?':>12}"
)
print(header)
print("-" * len(header))

results = {}
for name, X in feature_sets.items():
    # Modality (binary)
    mod_lr = cross_val_score(_lr_probe(), X, modality_labels, cv=5, scoring="accuracy")
    mod_mlp = cross_val_score(_mlp_probe(), X, modality_labels, cv=5, scoring="accuracy")
    mod_lr_acc, mod_mlp_acc = mod_lr.mean() * 100, mod_mlp.mean() * 100

    # Diagnosis (multiclass)
    diag_lr = cross_val_score(_lr_probe(multiclass=True), X, diagnosis_labels, cv=5, scoring="accuracy")
    diag_mlp = cross_val_score(_mlp_probe(), X, diagnosis_labels, cv=5, scoring="accuracy")
    diag_lr_acc, diag_mlp_acc = diag_lr.mean() * 100, diag_mlp.mean() * 100

    # Leakage flags use the max over LR/MLP — any probe that picks it up is a leak.
    mod_leak = ""
    diag_leak = ""
    if "Content" in name or "All dims" in name:
        mod_leak = "YES" if max(mod_lr_acc, mod_mlp_acc) > chance_modality + 10 else "no"
    if "Style" in name:
        diag_leak = "YES" if max(diag_lr_acc, diag_mlp_acc) > chance_diagnosis + 10 else "no"

    print(
        f"{name:<22} "
        f"{mod_lr_acc:>8.1f}%  {mod_mlp_acc:>8.1f}%  "
        f"{diag_lr_acc:>8.1f}%  {diag_mlp_acc:>8.1f}%  "
        f"{mod_leak:>12} {diag_leak:>12}"
    )

    results[name] = {
        "modality_lr": mod_lr_acc,
        "modality_mlp": mod_mlp_acc,
        "diagnosis_lr": diag_lr_acc,
        "diagnosis_mlp": diag_mlp_acc,
    }

# ══════════════════════════════════════════════════════════════════
# 2. ALIGNMENT & UNIFORMITY (Wang & Isola 2020)
# ══════════════════════════════════════════════════════════════════


def compute_alignment(x, y, alpha=2):
    """Alignment: expected ||f(x) - f(y)||^alpha for positive pairs."""
    x_norm = x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)
    y_norm = y / (np.linalg.norm(y, axis=1, keepdims=True) + 1e-8)
    return np.mean(np.sum((x_norm - y_norm) ** alpha, axis=1))


def compute_uniformity(x, t=2):
    """Uniformity: log of average pairwise Gaussian potential."""
    x_norm = x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)
    sq_dists = np.sum((x_norm[:, None] - x_norm[None, :]) ** 2, axis=2)
    mask = ~np.eye(len(x), dtype=bool)
    return np.log(np.mean(np.exp(-t * sq_dists[mask])))


print(f"\n{'='*70}")
print("ALIGNMENT & UNIFORMITY — Content Representations")
print("=" * 70)
print("(Lower is better for both. Alignment measures positive-pair closeness;")
print(" Uniformity measures how spread out features are on the hypersphere.)\n")

print(f"{'Feature set':<22} {'Alignment (↓)':>14} {'Uniformity (↓)':>16}")
print("-" * 55)

for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    subsets = []
    if lc["content_idx"]:
        subsets.append(
            (f"Content (L{lvl})", feats[t1_idx][:, lc["content_idx"]], feats[t2_idx][:, lc["content_idx_v1"]])
        )
    if lc["style_idx"]:
        subsets.append((f"Style (L{lvl})", feats[t1_idx][:, lc["style_idx"]], feats[t2_idx][:, lc["style_idx_v1"]]))
    subsets.append((f"All dims (L{lvl})", lc["t1"], lc["t2"]))
    for name, t1_feat, t2_feat in subsets:
        align = compute_alignment(t1_feat, t2_feat)
        combined = np.vstack([t1_feat, t2_feat])
        if len(combined) > 500:
            rng = np.random.RandomState(42)
            sub_idx = rng.choice(len(combined), 500, replace=False)
            combined = combined[sub_idx]
        unif = compute_uniformity(combined)
        print(f"{name:<22} {align:>14.4f} {unif:>16.4f}")

# ══════════════════════════════════════════════════════════════════
# 3. VISUALIZATION — LR vs MLP, grouped bars
# ══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, max(5, 0.35 * len(results) + 2)))

names = list(results.keys())
y_pos = np.arange(len(names))
h = 0.38
colors_base = ["tab:blue" if "Content" in n else "tab:orange" if "Style" in n else "tab:gray" for n in names]

for ax, target, chance, title in [
    (axes[0], "modality", chance_modality, "Predict MODALITY (T1 vs T2)\nContent should be at chance"),
    (axes[1], "diagnosis", chance_diagnosis, "Predict DIAGNOSIS\nStyle should be at chance"),
]:
    lr_vals = [results[n][f"{target}_lr"] for n in names]
    mlp_vals = [results[n][f"{target}_mlp"] for n in names]
    ax.barh(y_pos - h / 2, lr_vals, h, label="Linear (LR)", color=colors_base, alpha=0.9)
    ax.barh(y_pos + h / 2, mlp_vals, h, label="Non-linear (MLP)", color=colors_base, alpha=0.5, hatch="//")
    ax.axvline(chance, color="red", ls="--", label=f"Chance ({chance:.1f}%)")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel("Accuracy (%)")
    ax.set_title(title)
    ax.set_xlim(0, 100)
    ax.legend(loc="lower right", fontsize=8)

plt.suptitle("Content–Style Information Leakage Test — LR vs MLP", fontsize=14)
plt.tight_layout()
plt.savefig("content_style_leakage.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✓ Saved content_style_leakage.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSIS PROBE BREAKDOWN — step 1 of the content/style diagnostic
# Compare content-only, style-only, and concat(content, style) for
# predicting diagnosis, aggregated across levels. Balanced accuracy +
# macro-AUC so class imbalance doesn't confound the comparison.
# Reuses `all_features`, `level_content`, `_per_view_select`, `all_labels`
# defined in the leakage cell above.
# ══════════════════════════════════════════════════════════════════
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, StandardScaler

diag_labels = np.asarray(all_labels)
n_classes_d = len(np.unique(diag_labels))
chance_diag = 1.0 / n_classes_d


def _probe():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, C=1.0, random_state=42, class_weight="balanced", multi_class="multinomial"),
    )


cv_d = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Aggregate content-only and style-only features across all levels
content_parts, style_parts = [], []
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    if lc["content_idx"]:
        content_parts.append(_per_view_select(feats, lc["content_idx"], lc["content_idx_v1"]))
    if lc["style_idx"]:
        style_parts.append(_per_view_select(feats, lc["style_idx"], lc["style_idx_v1"]))

X_content = np.concatenate(content_parts, axis=1) if content_parts else None
X_style = np.concatenate(style_parts, axis=1) if style_parts else None
X_both = np.concatenate([X_content, X_style], axis=1) if (X_content is not None and X_style is not None) else None

# Assemble the feature sets to probe — per-level plus aggregated
feature_sets = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    if lc["content_idx"]:
        feature_sets[f"Content L{lvl}"] = _per_view_select(feats, lc["content_idx"], lc["content_idx_v1"])
    if lc["style_idx"]:
        feature_sets[f"Style L{lvl}"] = _per_view_select(feats, lc["style_idx"], lc["style_idx_v1"])
if X_content is not None:
    feature_sets["Content (all levels)"] = X_content
if X_style is not None:
    feature_sets["Style (all levels)"] = X_style
if X_both is not None:
    feature_sets["Content ⊕ Style"] = X_both

print("=" * 82)
print("DIAGNOSIS PROBE BREAKDOWN")
print(
    f"Chance balAcc = {chance_diag*100:.1f}% ({n_classes_d} classes), "
    f"N = {len(diag_labels)}, 5-fold stratified CV, balanced class weights"
)
print("=" * 82)
print(f"{'Feature set':<25} {'Dim':>5} {'BalAcc':>12} {'AUC (macro OvR)':>20}")
print("-" * 82)

probe_results = {}
for name, X in feature_sets.items():
    bal = cross_val_score(_probe(), X, diag_labels, cv=cv_d, scoring="balanced_accuracy")
    try:
        auc = cross_val_score(_probe(), X, diag_labels, cv=cv_d, scoring="roc_auc_ovr_weighted")
    except Exception:
        auc = np.array([np.nan])
    probe_results[name] = {
        "bal": bal.mean(),
        "bal_std": bal.std(),
        "auc": auc.mean(),
        "auc_std": auc.std(),
        "dim": X.shape[1],
    }
    print(
        f"{name:<25} {X.shape[1]:>5d} "
        f"{bal.mean()*100:>7.1f}% ±{bal.std()*100:>3.1f} "
        f"{auc.mean():>12.3f} ±{auc.std():>5.3f}"
    )

# ── Decision table ──────────────────────────────────────────────
print()
print("=" * 82)
print("INTERPRETATION")
print("=" * 82)
A = probe_results.get("Content (all levels)")
B = probe_results.get("Style (all levels)")
C = probe_results.get("Content ⊕ Style")
if A and B and C:
    gap_sc = B["bal"] - A["bal"]
    gain_both = C["bal"] - max(A["bal"], B["bal"])
    print(f"  Content  balAcc = {A['bal']*100:5.1f}%   (chance {chance_diag*100:.1f}%)")
    print(f"  Style    balAcc = {B['bal']*100:5.1f}%")
    print(f"  Both     balAcc = {C['bal']*100:5.1f}%")
    print(f"  Style − Content : {gap_sc*100:+.1f} pp")
    print(f"  Both   − max(A,B): {gain_both*100:+.1f} pp")
    print()
    content_near_chance = (A["bal"] - chance_diag) < 0.03
    if content_near_chance:
        print("  → Content near chance. Most likely: InfoNCE isn't pushing diagnosis")
        print("    into content. Next: decode-space probe (step 2).")
    elif gap_sc > 0.05 and gain_both > 0.03:
        print("  → Content and style carry DIFFERENT diagnostic info; combining helps.")
        print("    Real disentanglement mismatch. Next: step 2 (decode-space probe).")
    elif gap_sc > 0.05 and gain_both < 0.02:
        print("  → Style ≫ Content but combining adds nothing — same signal, wrong stream.")
        print("    Capacity/pressure issue. Next: swap-consistency loss or shrink style.")
    else:
        print("  → Content ≈ Style. Not really a disentanglement failure.")
        print("    Low priority; focus elsewhere or tighten the probe.")
else:
    print("  (need content, style, and both feature sets — check level_content indices)")

# ── Visualisation ──────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(9, 0.4 * len(probe_results) + 2))
names = list(probe_results.keys())
bals = [probe_results[n]["bal"] * 100 for n in names]
errs = [probe_results[n]["bal_std"] * 100 for n in names]
colors = [
    "tab:blue"
    if "Content" in n and "Style" not in n
    else "tab:orange"
    if "Style" in n and "Content" not in n
    else "tab:green"
    for n in names
]
ax.barh(names, bals, xerr=errs, color=colors, alpha=0.85)
ax.axvline(chance_diag * 100, color="red", ls="--", label=f"Chance ({chance_diag*100:.1f}%)")
ax.set_xlabel("Balanced accuracy (%)")
ax.set_title("Diagnosis probe — content vs style vs concat")
ax.set_xlim(0, 100)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("diagnosis_probe_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✓ Saved diagnosis_probe_breakdown.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSIS PROBE — follow-ups (run after the breakdown cell)
# 1. AD-vs-CN binary (drop MCI, the class that wrecks 3-way balAcc)
# 2. Nonlinear probes (MLP, RandomForest) to rule out probe capacity
# 3. Raw-volume baseline — the achievable ceiling on this N
#
# Reuses: X_content, X_style, X_both, diag_labels, _probe, cv_d, label_map,
#         items, val_transforms, all_modalities, all_subjects
#
# Note: `diag_labels` / `all_features` rows are interleaved per subject
# (subj0_T1, subj0_T2, subj1_T1, subj1_T2, ...). `X_raw` must follow the
# same ordering.
# ══════════════════════════════════════════════════════════════════
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
import torch.nn.functional as F

# ── Label distribution sanity ──────────────────────────────────
_uniq, _counts = np.unique(diag_labels, return_counts=True)
print("Class distribution (int → count):")
for _u, _c in zip(_uniq.tolist(), _counts.tolist()):
    _nm = label_names.get(_u, "?") if "label_names" in dir() else "?"
    print(f"  {_u} ({_nm}): {_c}")


# Resolve AD / CN integer labels
def _lookup(name):
    for k, v in label_map.items():
        if str(k).upper() == name:
            return v
    return None


ad_label = _lookup("AD")
cn_label = _lookup("CN")
assert ad_label is not None and cn_label is not None, f"Could not find AD/CN in {label_map}"

# Groups for CV — one group per subject so T1 and T2 of the same subject
# can't leak across folds. Fall back to row index if subjects aren't tracked.
try:
    groups_all = np.asarray(all_subjects)
    use_groups = len(groups_all) == len(diag_labels) and len(np.unique(groups_all)) > 1
except NameError:
    use_groups = False

if use_groups:
    cv_d_grp = GroupKFold(n_splits=5)
    print(f"\nUsing GroupKFold (5 splits) on subject IDs — {len(np.unique(groups_all))} unique subjects.")
else:
    cv_d_grp = cv_d
    groups_all = None
    print("\n⚠ all_subjects not available — using StratifiedKFold (within-subject T1/T2 may leak).")


def _score(clf, X, y, scoring, groups=None, cv=None):
    cv = cv if cv is not None else (cv_d_grp if use_groups else cv_d)
    return cross_val_score(clf, X, y, cv=cv, scoring=scoring, groups=groups)


# ══════════════════════════════════════════════════════════════════
# 1. AD vs CN binary probe
# ══════════════════════════════════════════════════════════════════
bin_mask = (diag_labels == ad_label) | (diag_labels == cn_label)
y_bin = (diag_labels[bin_mask] == ad_label).astype(int)
groups_bin = groups_all[bin_mask] if use_groups else None
cv_bin = GroupKFold(n_splits=5) if use_groups else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(
    f"\nBinary AD-vs-CN: N={bin_mask.sum()}  AD={int(y_bin.sum())}  "
    f"CN={int((1 - y_bin).sum())}  chance balAcc = 50.0%"
)

bin_sets = {}
if X_content is not None:
    bin_sets["Content (all)"] = X_content[bin_mask]
if X_style is not None:
    bin_sets["Style (all)"] = X_style[bin_mask]
if X_both is not None:
    bin_sets["Both"] = X_both[bin_mask]

print(f"\n{'Feature set':<16} {'LR balAcc':>12} {'LR AUC':>10}")
print("-" * 42)
for name, X in bin_sets.items():
    bal = cross_val_score(_probe(), X, y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=groups_bin)
    auc = cross_val_score(_probe(), X, y_bin, cv=cv_bin, scoring="roc_auc", groups=groups_bin)
    print(f"{name:<16} {bal.mean()*100:>8.1f}% ±{bal.std()*100:.1f} " f"{auc.mean():>7.3f} ±{auc.std():.3f}")


# ══════════════════════════════════════════════════════════════════
# 2. Nonlinear probes (3-class) on content / style / both
# ══════════════════════════════════════════════════════════════════
def _mlp():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42, early_stopping=True, alpha=1e-3),
    )


def _rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)


print("\n" + "=" * 90)
print("NONLINEAR PROBE — does signal exist but not linearly?")
print("=" * 90)
print(
    f"{'Feature set':<16} {'3-cls LR':>11} {'3-cls MLP':>12} {'3-cls RF':>11} "
    f"{'AD-CN LR':>11} {'AD-CN MLP':>12} {'AD-CN RF':>11}"
)
print("-" * 90)
for name, X in [("Content (all)", X_content), ("Style (all)", X_style), ("Both", X_both)]:
    if X is None:
        continue
    g3 = groups_all if use_groups else None
    gb = groups_bin if use_groups else None

    lr3 = cross_val_score(_probe(), X, diag_labels, cv=cv_d_grp, scoring="balanced_accuracy", groups=g3).mean()
    mlp3 = cross_val_score(_mlp(), X, diag_labels, cv=cv_d_grp, scoring="balanced_accuracy", groups=g3).mean()
    rf3 = cross_val_score(_rf(), X, diag_labels, cv=cv_d_grp, scoring="balanced_accuracy", groups=g3).mean()

    Xb = X[bin_mask]
    lrb = cross_val_score(_probe(), Xb, y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=gb).mean()
    mlpb = cross_val_score(_mlp(), Xb, y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=gb).mean()
    rfb = cross_val_score(_rf(), Xb, y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=gb).mean()
    print(
        f"{name:<16} {lr3*100:>8.1f}% {mlp3*100:>10.1f}% {rf3*100:>9.1f}% "
        f"{lrb*100:>8.1f}% {mlpb*100:>10.1f}% {rfb*100:>9.1f}%"
    )

# ══════════════════════════════════════════════════════════════════
# 3. Raw-volume baseline — ceiling for this dataset
# ══════════════════════════════════════════════════════════════════
# Avg-pool each modality to an 8^3 grid plus a few global stats.
# Emit rows in the SAME interleaved order as `all_features`:
#   [subj0_T1, subj0_T2, subj1_T1, subj1_T2, ...]
# so labels/groups line up 1:1.
print("\n" + "=" * 78)
print("RAW-VOLUME BASELINE (pooled voxels + global stats, no learned features)")
print("=" * 78)
print(f"Collecting raw features for {len(items)} subjects (T1 + T2 each)...")


def _vol_features(img):
    """img: (1, 1, D, H, W) float tensor on CPU."""
    with torch.no_grad():
        pooled = F.adaptive_avg_pool3d(img, (8, 8, 8)).flatten().numpy()
    brain_vox = img[img > 0]
    if brain_vox.numel() > 1:
        q_low = brain_vox.quantile(0.1)
        q_high = brain_vox.quantile(0.9)
        stats = np.array(
            [
                float((img > 0).float().sum()),
                float(brain_vox.mean()),
                float(brain_vox.std()),
                float((brain_vox < q_low).float().mean()),
                float((brain_vox > q_high).float().mean()),
            ]
        )
    else:
        stats = np.zeros(5)
    return np.concatenate([pooled, stats])


raw_feats = []
for i, it in enumerate(items):
    t = val_transforms({"image_t1": it["image"], "image_t2": it.get("z_image", it["image"])})
    for key in ("image_t1", "image_t2"):
        raw_feats.append(_vol_features(t[key].unsqueeze(0).float()))
    if (i + 1) % 50 == 0 or (i + 1) == len(items):
        print(f"  {i+1}/{len(items)}")

X_raw = np.stack(raw_feats)
print(f"Raw feature shape: {X_raw.shape}  (expected ({len(diag_labels)}, ...))")
assert X_raw.shape[0] == len(diag_labels), "X_raw rows must match diag_labels"

print(f"\n{'Probe':<6} {'3-cls balAcc':>14} {'AD-CN balAcc':>16} {'AD-CN AUC':>12}")
print("-" * 52)
for name, mk in [("LR", _probe), ("MLP", _mlp), ("RF", _rf)]:
    g3 = groups_all if use_groups else None
    gb = groups_bin if use_groups else None
    bal3 = cross_val_score(mk(), X_raw, diag_labels, cv=cv_d_grp, scoring="balanced_accuracy", groups=g3).mean()
    balb = cross_val_score(mk(), X_raw[bin_mask], y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=gb).mean()
    try:
        aucb = cross_val_score(mk(), X_raw[bin_mask], y_bin, cv=cv_bin, scoring="roc_auc", groups=gb).mean()
    except Exception:
        aucb = float("nan")
    print(f"{name:<6} {bal3*100:>11.1f}%  {balb*100:>13.1f}%  {aucb:>11.3f}")

# ══════════════════════════════════════════════════════════════════
# Summary interpretation
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 78)
print("SUMMARY — how to read the numbers")
print("=" * 78)
print("  • Raw AD-vs-CN balAcc ≈ 50% → dataset/split isn't diagnosable at all.")
print("    Not a model problem. Revisit labels, N, or split.")
print("  • Raw ≫ Content, MLP ≈ LR → model is discarding signal that's in the image.")
print("    Likely capacity or objective issue; step 3 (adversary run) relevant.")
print("  • MLP ≫ LR on same features → signal is there but nonlinearly encoded.")
print("    Not a disentanglement problem; use a stronger downstream probe.")
print("  • Everything at chance including raw → your 400-subject split is underpowered")
print("    or MCI is dominating; AD-vs-CN column is the more reliable signal here.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSTIC — reconcile notebook probe vs. training-time probe
# ══════════════════════════════════════════════════════════════════
# (A) Check whether the Gumbel content mask drifts across samples.
#     Notebook probes record the FIRST sample's mask only; if the mask
#     is stochastic in eval mode, the recorded indices are wrong for
#     the rest of the dataset and the probe result is untrustworthy.
# (B) Call eval.cross_reconstruction._collect_representations +
#     _metrics_for_level directly on the notebook's data, so the
#     only thing that can differ from the training metric is the
#     underlying sample set and transforms.
import sys
import pathlib
from types import SimpleNamespace

_ROOT = pathlib.Path().resolve()
while _ROOT != _ROOT.parent and not (_ROOT / "eval" / "cross_reconstruction.py").exists():
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from eval.cross_reconstruction import _collect_representations, _metrics_for_level

# ─────────────────────────────────────────────────────────────────
# (A) Mask drift across samples
# ─────────────────────────────────────────────────────────────────
print("=" * 70)
print("DIAGNOSTIC A — Gumbel content mask stability")
print("=" * 70)
print(f"vqvae_model.training = {vqvae_model.training}  (False = deterministic path expected)")

_n_probe = min(20, len(items))
_per_level_masks = {lvl: [] for lvl in range(nb_levels)}
for i in range(_n_probe):
    t = val_transforms({"image_t1": items[i]["image"], "image_t2": items[i]["z_image"]})
    img = t["image_t1"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _, _, _, _, _, _, soft_masks, *_ = vqvae_model(img, return_recon=False, pool_only=True, view_idx=0)
    for lvl, m in soft_masks.items():
        m_t = m[0] if isinstance(m, tuple) else m
        _per_level_masks[lvl].append(tuple(torch.where(m_t.bool())[-1].tolist()))

for lvl, masks in _per_level_masks.items():
    if not masks:
        continue
    first = masks[0]
    drift = sum(1 for m in masks if m != first)
    unique = len(set(masks))
    flag = "  ⚠ STOCHASTIC (probe indices unreliable)" if drift else "  ✓ deterministic"
    print(f"  L{lvl}: {drift}/{len(masks)} samples differ from first; {unique} unique mask(s){flag}")

# ─────────────────────────────────────────────────────────────────
# (B) Training-time probe, run inline on the notebook's samples
# ─────────────────────────────────────────────────────────────────
print()
print("=" * 70)
print("DIAGNOSTIC B — training-time probe on the notebook's sample set")
print("=" * 70)

_BATCH_SIZE = 4


def _paired_batches(_items, _transforms, _batch_size):
    """Yield {'image': [t1_batch, t2_batch]} dicts, mimicking the training val loader."""
    buf_t1, buf_t2 = [], []
    for it in _items:
        t = _transforms({"image_t1": it["image"], "image_t2": it["z_image"]})
        buf_t1.append(t["image_t1"])
        buf_t2.append(t["image_t2"])
        if len(buf_t1) == _batch_size:
            yield {"image": [torch.stack(buf_t1), torch.stack(buf_t2)]}
            buf_t1, buf_t2 = [], []
    if buf_t1:
        yield {"image": [torch.stack(buf_t1), torch.stack(buf_t2)]}


_args = SimpleNamespace(
    content_style_levels=list(range(nb_levels)),
    # Fallback only — _collect_representations uses soft_content_masks when present.
    content_indices=[list(all_content_indices.get(lvl, [])) for lvl in range(nb_levels)],
)

reps = _collect_representations(
    vqvae_model,
    _paired_batches(items, val_transforms, _BATCH_SIZE),
    _args,
    DEVICE,
    max_batches=10**6,
)

print(f"Levels collected: {sorted(reps.keys())}")
for lvl, r in reps.items():
    print(
        f"  L{lvl}: content_v0 {r['content_v0'].shape}, content_v1 {r['content_v1'].shape}, "
        f"style_v0 {r['style_v0'].shape}, style_v1 {r['style_v1'].shape}"
    )

print()
print("Training-probe metrics (apples-to-apples with eval/cross_reconstruction.py):")
print(f"{'Level':<8} {'modality_probe_acc':>22} {'modality_invariance':>22}")
print("-" * 54)
for lvl in sorted(reps.keys()):
    m = _metrics_for_level(reps[lvl])
    print(f"  L{lvl:<5} {m['content/modality_probe_acc']:>22.3f} " f"{m['content/modality_invariance']:>22.3f}")

print()
print("If DIAGNOSTIC B matches the training log (~0.6) but the cell above still")
print("reports ~0.8, the gap is purely how the notebook assembles features")
print("(single-view forward, first-sample-only mask, stochastic Gumbel, etc).")
print("If DIAGNOSTIC B ALSO reports ~0.8, the gap is the sample set / checkpoint,")
print("not a code bug — compare `items` here to the val split used during training.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSTIC C — sweep sample size N vs. modality probe accuracy
# ══════════════════════════════════════════════════════════════════
# Logistic-regression probe accuracy grows with N. If the training log's
# ~0.6 was simply under-resolved (small val N) and the true leakage is
# ~0.8, this sweep will show the curve rising from ~0.6 at small N to
# ~0.8 as N grows. If accuracy is roughly flat across N, the gap is NOT
# sample size (→ suspect split/checkpoint/transforms instead).

import numpy as np
import matplotlib.pyplot as plt

_N_total = len(items)
_N_grid = sorted({n for n in [25, 50, 100, 200, 400, _N_total] if n <= _N_total and n >= 10})
print(f"Sweeping N ∈ {_N_grid}  (total available: {_N_total})")

_rng = np.random.RandomState(0)
_perm = _rng.permutation(_N_total)  # fixed shuffle so subsets are nested-ish, not modality-ordered

_sweep = {lvl: [] for lvl in range(nb_levels)}

for _N in _N_grid:
    _sub_items = [items[i] for i in _perm[:_N]]
    _reps_N = _collect_representations(
        vqvae_model,
        _paired_batches(_sub_items, val_transforms, _BATCH_SIZE),
        _args,
        DEVICE,
        max_batches=10**6,
    )
    for lvl in _reps_N:
        m = _metrics_for_level(_reps_N[lvl])
        _sweep[lvl].append((m["content/modality_probe_acc"], m["content/modality_invariance"]))
    # Report progress
    line = f"  N={_N:>4}"
    for lvl in sorted(_reps_N.keys()):
        line += f"   L{lvl}: {_sweep[lvl][-1][0]:.3f}"
    print(line)

# ── Plot ──
fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))
for lvl, pts in _sweep.items():
    if not pts:
        continue
    accs = [p[0] for p in pts]
    ax.plot(_N_grid, accs, marker="o", label=f"Level {lvl}")
ax.axhline(0.5, color="gray", ls=":", label="chance")
ax.set_xscale("log")
ax.set_xlabel("N (paired subjects)")
ax.set_ylabel("Content → modality probe accuracy")
ax.set_title("Modality-leakage probe vs. sample size\n(matches training probe code; varies N)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("modality_probe_vs_N.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved modality_probe_vs_N.png")

print()
print("Interpretation:")
print("  • Monotone rise → the training log's low number was under-resolved;")
print("    the true leakage at this checkpoint is the high-N value.")
print("  • Flat curve    → gap is NOT sample size. Check split / checkpoint /")
print("    transforms (diagnostic B already rules out code-path differences).")

## 8. Reconstruction visualisation

Compare original slices with their VQ-VAE-2 reconstructions.

## 9. Diagnostic label prediction

For each feature set (all encoder levels + level-0 content/style subsets) we train a
**logistic regression** and a **random forest** classifier on T1 features only (so modality
doesn't leak into the score), using stratified 5-fold cross-validation.

This tells us:
- **Which hierarchical level** encodes the most diagnostically-relevant information
- **Whether content or style channels** at level 0 carry the diagnostic signal
- How close any level is to chance (baseline = majority-class rate)

Results are shown as a bar chart and a summary table.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier

# ── use T1 features only so modality can't trivially distinguish classes ──────
t1_mask = all_modalities == "T1"
y = all_labels[t1_mask]

# Build the feature sets to evaluate
feature_sets = {}
for i in range(nb_levels):
    feature_sets[f"Level {i} (all dims)"] = all_features[f"level_{i}"][t1_mask]

# Level-0 content / style subsets
if 0 in all_content_indices and len(all_style_indices.get(0, [])) > 0:
    level0_feats = all_features["level_0"]
    feature_sets["Level 0 — content"] = level0_feats[t1_mask][:, all_content_indices[0]]
    feature_sets["Level 0 — style"] = level0_feats[t1_mask][:, all_style_indices[0]]


# ── classifiers ───────────────────────────────────────────────────────────────
def make_lr():
    """Logistic Regression with standard scaling."""
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
        ]
    )


def make_rf():
    """Random Forest (no scaling needed)."""
    return RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
chance = cross_val_score(DummyClassifier(strategy="most_frequent"), np.zeros((len(y), 1)), y, cv=cv).mean()

print(f"{'Feature set':<28} {'LR acc':>8} {'LR ±':>7}  {'RF acc':>8} {'RF ±':>7}")
print("-" * 65)

results = []
for name, X in feature_sets.items():
    lr_scores = cross_val_score(make_lr(), X, y, cv=cv, scoring="accuracy")
    rf_scores = cross_val_score(make_rf(), X, y, cv=cv, scoring="accuracy")
    results.append(
        {
            "Feature set": name,
            "LR mean": lr_scores.mean(),
            "LR std": lr_scores.std(),
            "RF mean": rf_scores.mean(),
            "RF std": rf_scores.std(),
        }
    )
    print(
        f"{name:<28} {lr_scores.mean():.3f}   ±{lr_scores.std():.3f}   "
        f"{rf_scores.mean():.3f}   ±{rf_scores.std():.3f}"
    )

print("-" * 65)
print(f"{'Majority-class baseline':<28} {chance:.3f}")

results_df = pd.DataFrame(results)
print("\nSummary dataframe saved as  results_df  (use .to_csv() to export)")

In [ ]:
# ── Bar chart: LR and RF accuracy per feature set ────────────────────────────
x = np.arange(len(results_df))
width = 0.35

fig, ax = plt.subplots(figsize=(max(8, len(results_df) * 1.6), 5))

bars_lr = ax.bar(
    x - width / 2,
    results_df["LR mean"],
    width,
    yerr=results_df["LR std"],
    capsize=4,
    label="Logistic Regression",
    color="steelblue",
    alpha=0.85,
)

bars_rf = ax.bar(
    x + width / 2,
    results_df["RF mean"],
    width,
    yerr=results_df["RF std"],
    capsize=4,
    label="Random Forest",
    color="tomato",
    alpha=0.85,
)

# Chance-level line
ax.axhline(chance, color="grey", linestyle="--", linewidth=1.2, label=f"Chance ({chance:.2f})")

# Annotate bars with their mean accuracy
for bar in list(bars_lr) + list(bars_rf):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005, f"{h:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(results_df["Feature set"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("5-fold CV accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Diagnostic label prediction accuracy per VQ-VAE-2 feature set\n(T1 features only, 5-fold stratified CV)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("diagnostic_prediction_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved diagnostic_prediction_accuracy.png")

# ── Per-class breakdown (confusion matrix for best feature set) ──────────────
best_name = results_df.loc[results_df["RF mean"].idxmax(), "Feature set"]
best_X = feature_sets[best_name]
print(f"\nBest feature set by RF accuracy: '{best_name}'")
print("Showing confusion matrix (RF, 5-fold average) ...")

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_pred = cross_val_predict(make_rf(), best_X, y, cv=cv)
cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[label_names[i] for i in sorted(label_names)])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion matrix — RF on '{best_name}'")
plt.tight_layout()
plt.savefig("confusion_matrix_best_level.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved confusion_matrix_best_level.png")

print("\nClassification report:")
print(classification_report(y, y_pred, target_names=[label_names[i] for i in sorted(label_names)]))

In [ ]:
N_SHOW = 3  # number of subjects to show

fig, axes = plt.subplots(N_SHOW, 4, figsize=(14, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]

for row, item in enumerate(items[:N_SHOW]):
    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for col_offset, (key, mod_label) in enumerate([("image_t1", "T1"), ("image_t2", "T2")]):
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)
        _vi = 0 if mod_label == "T1" else 1

        with torch.no_grad():
            recon, _, _, _, _, _, _, _ = vqvae_model(img, return_recon=True, pool_only=False, view_idx=_vi)

        # Mid-sagittal slice (dim 2 = D)
        mid = img.shape[2] // 2
        orig_slice = img[0, 0, mid].cpu().numpy()
        recon_slice = recon[0, 0, mid].cpu().float().numpy()

        axes[row, col_offset * 2].imshow(orig_slice, cmap="gray", origin="lower")
        axes[row, col_offset * 2].set_title(f"Subject {row} — {mod_label} original")
        axes[row, col_offset * 2].axis("off")

        axes[row, col_offset * 2 + 1].imshow(recon_slice, cmap="gray", origin="lower")
        axes[row, col_offset * 2 + 1].set_title(f"Subject {row} — {mod_label} reconstruction")
        axes[row, col_offset * 2 + 1].axis("off")

plt.suptitle("VQ-VAE-2 Reconstructions (mid-sagittal)", fontsize=13)
plt.tight_layout()
plt.savefig("reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved reconstructions.png")

## 10. Save extracted features

In [ ]:
save_dict = {
    "labels": all_labels,
    "modalities": all_modalities,
    "label_map_keys": np.array(list(label_map.keys())),
    "label_map_values": np.array(list(label_map.values())),
    "checkpoint_step": np.array([checkpoint["step"]]),
    "content_indices": np.array(all_content_indices.get(0, [])) if 0 in all_content_indices else np.array([]),
}
for i in range(nb_levels):
    save_dict[f"features_level_{i}"] = all_features[f"level_{i}"]

np.savez("extracted_latents.npz", **save_dict)
print("✓ Saved extracted_latents.npz")
for i in range(nb_levels):
    print(f"  level_{i}: {all_features[f'level_{i}'].shape}")

## 11. Codebook Analysis — Extract Indices

Run a separate forward pass with `return_recon=True` so that non-coarsest codebook
levels get proper decoder conditioning (without it they receive zero-padded input
and produce incorrect indices).

We collect:
- **Per-level codebook usage histograms** (normalized frequency of each entry per subject)
- **Raw codebook indices** for the code-replacement experiment later

In [ ]:
nb_entries = vqvae_module.codebooks[0].n_embed
print(f"Codebook: {nb_levels} levels, {nb_entries} entries each\n")

# Storage: index 0 = finest (level 0), index nb_levels-1 = coarsest
cb_histograms = [[] for _ in range(nb_levels)]  # per-subject normalized histograms
cb_labels = []  # diagnosis label per sample
cb_modalities = []  # "T1" or "T2"

# Per-class examples for code-replacement comparison (CN vs AD)
class_examples = {}

# Position-wise code frequency accumulators (T1 only).
# pos_counts[level] has shape (D_l, H_l, W_l, nb_entries) — lazily initialized.
pos_counts = [None for _ in range(nb_levels)]
n_t1_subjects = 0  # number of T1 subjects contributing to pos_counts

print(f"Extracting codebook indices from {len(items)} subjects (T1 + T2) ...")
print(f"Will keep one T1 example per class for code-replacement demo: {list(label_names.values())}")

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)
        _vi = 0 if modality == "T1" else 1

        with torch.no_grad():
            recon, diffs, _, _, _, id_outputs_raw, _, _ = vqvae_model(
                img,
                return_recon=True,
                pool_only=False,
                view_idx=_vi,
            )

        # id_outputs_raw is coarsest-first; reverse to finest-first
        id_outputs = id_outputs_raw[::-1]

        for lvl in range(nb_levels):
            hist = torch.bincount(id_outputs[lvl].reshape(-1), minlength=nb_entries)
            hist = hist.float() / hist.sum()
            cb_histograms[lvl].append(hist.cpu().numpy())

        cb_labels.append(item["label"])
        cb_modalities.append(modality)

        # Accumulate position-wise code counts (T1 only to avoid modality confound)
        if modality == "T1":
            n_t1_subjects += 1
            for lvl in range(nb_levels):
                ids_lvl = id_outputs[lvl].squeeze(0).cpu()  # (D_l, H_l, W_l)
                if pos_counts[lvl] is None:
                    spatial = ids_lvl.shape
                    pos_counts[lvl] = torch.zeros(*spatial, nb_entries, dtype=torch.long)
                # Scatter-add: for each position, increment the count for the observed code
                pos_counts[lvl].scatter_(-1, ids_lvl.unsqueeze(-1), 1)

        # Keep one T1 example per diagnosis class
        if modality == "T1":
            class_name = label_names[item["label"]]
            if class_name not in class_examples:
                t1_tensor = transformed[key]
                ex_affine = None
                if hasattr(t1_tensor, "affine"):
                    ex_affine = t1_tensor.affine.numpy()
                elif hasattr(t1_tensor, "meta") and "affine" in t1_tensor.meta:
                    ex_affine = t1_tensor.meta["affine"].numpy()

                class_examples[class_name] = {
                    "id_outputs": [id_outputs[lvl].clone() for lvl in range(nb_levels)],
                    "image": img.clone(),
                    "item": item,
                    "affine": ex_affine,
                }
                print(f"  ✓ Stored {class_name} example: subject {item.get('subject', idx)}")

# Stack
for lvl in range(nb_levels):
    cb_histograms[lvl] = np.array(cb_histograms[lvl])
cb_labels = np.array(cb_labels)
cb_modalities = np.array(cb_modalities)

# Backward compat
_fallback_class = list(class_examples.keys())[-1] if class_examples else None
last_id_outputs = class_examples[_fallback_class]["id_outputs"] if _fallback_class else None
last_image = class_examples[_fallback_class]["image"] if _fallback_class else None
last_item = class_examples[_fallback_class]["item"] if _fallback_class else None
last_affine = class_examples[_fallback_class]["affine"] if _fallback_class else None

print(f"\n✓ Codebook extraction complete ({cb_histograms[0].shape[0]} samples)")
print(f"  T1 subjects for position-wise analysis: {n_t1_subjects}")
for lvl in range(nb_levels):
    used = (cb_histograms[lvl].sum(axis=0) > 0).sum()
    sp = tuple(pos_counts[lvl].shape[:-1]) if pos_counts[lvl] is not None else "?"
    print(
        f"  Level {lvl}: histogram shape {cb_histograms[lvl].shape}, "
        f"codes used: {used}/{nb_entries}, code map spatial: {sp}"
    )
print(f"  Class examples stored: {list(class_examples.keys())}")

## 12-pre. Style Codebook Perplexity (capacity diagnostic)

If modality invariance fails at a given level while levels above succeed, one hypothesis is that the **style codebook at that level is capacity-starved** — it cannot represent the modality contrast, so modality info leaks back into content. This cell measures, per level:

- **Codebook size** `K`
- **Unique codes used** across all spatial positions / subjects
- **Perplexity** `exp(-Σ p_k log p_k)` of the marginal usage distribution (effective number of codes in use)
- **Utilization** = perplexity / K ∈ [0, 1]

Split by modality (T1 vs T2): if style is doing its job, T1 and T2 should concentrate on different codes (low Jensen–Shannon overlap). If the codebook is saturated (utilization near 1) **and** the per-level modality probe on style features is already high, the level is capacity-starved → **problem 3** (increase `--style-nb-entries` / style channels). If utilization is low / codebook is barely used, the issue is routing/pressure, not capacity → **problem 1**.


In [ ]:
# ── Style codebook perplexity per level ───────────────────────────
# Requires quantize_style=True. Reuses `items` / `val_transforms` / `DEVICE`
# from the content-codebook extraction cell above.

if not getattr(vqvae_module, "quantize_style", False):
    print("⚠ quantize_style is False — no style codebooks to analyze. Skipping.")
else:
    style_levels = sorted(vqvae_module.style_codebooks.keys(), key=int)
    style_sizes = {int(l): vqvae_module.style_nb_entries_per_level[int(l)] for l in style_levels}
    print(f"Style codebooks: levels {[int(l) for l in style_levels]}")
    for l in style_levels:
        print(f"  level {l}: K = {style_sizes[int(l)]}")
    print()

    # Accumulate counts per (level, modality)
    style_counts = {int(l): {"T1": None, "T2": None} for l in style_levels}

    def _to_counts(ids_flat, K):
        return torch.bincount(ids_flat, minlength=K).cpu().numpy().astype(np.int64)

    print(f"Running forward passes on {len(items)} subjects (T1 + T2) ...")
    for idx, item in enumerate(items):
        if idx % 20 == 0:
            print(f"  {idx}/{len(items)} ...")
        data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
        transformed = val_transforms(data_dict)
        for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
            img = transformed[key].unsqueeze(0).to(DEVICE)
            _vi = 0 if modality == "T1" else 1
            with torch.no_grad():
                out = vqvae_model(img, return_recon=True, pool_only=False, view_idx=_vi)
            # 8-tuple: (..., style_id_outputs)
            style_ids = out[7]  # dict {lvl: LongTensor indices}
            for lvl, ids in style_ids.items():
                K = style_sizes[lvl]
                c = _to_counts(ids.reshape(-1), K)
                cur = style_counts[lvl][modality]
                style_counts[lvl][modality] = c if cur is None else cur + c

    # ── Perplexity / utilization per level ────────────────────────
    def _perplexity(counts):
        total = counts.sum()
        if total == 0:
            return 0.0, 0, np.zeros_like(counts, dtype=np.float64)
        p = counts.astype(np.float64) / total
        nz = p > 0
        H = -(p[nz] * np.log(p[nz])).sum()
        return float(np.exp(H)), int(nz.sum()), p

    def _jsd(p, q):
        m = 0.5 * (p + q)

        def _kl(a, b):
            mask = (a > 0) & (b > 0)
            return float((a[mask] * np.log(a[mask] / b[mask])).sum())

        return 0.5 * _kl(p, m) + 0.5 * _kl(q, m)

    print()
    print("=" * 88)
    print(f"{'Level':<6} {'K':>5} {'modality':>9} {'used':>6} {'perplex':>9} {'util':>7}  {'JSD(T1,T2)':>12}")
    print("-" * 88)
    for lvl in sorted(style_counts.keys()):
        K = style_sizes[lvl]
        c_t1 = style_counts[lvl]["T1"]
        c_t2 = style_counts[lvl]["T2"]
        c_all = (c_t1 if c_t1 is not None else 0) + (c_t2 if c_t2 is not None else 0)

        ppl_all, used_all, p_all = _perplexity(c_all)
        ppl_t1, used_t1, p_t1 = _perplexity(c_t1) if c_t1 is not None else (0.0, 0, None)
        ppl_t2, used_t2, p_t2 = _perplexity(c_t2) if c_t2 is not None else (0.0, 0, None)

        jsd = _jsd(p_t1, p_t2) if (p_t1 is not None and p_t2 is not None) else float("nan")

        print(f"{lvl:<6} {K:>5} {'all':>9} {used_all:>6} {ppl_all:>9.2f} {ppl_all/K:>7.2%}  {jsd:>12.4f}")
        print(f"{'':<6} {'':>5} {'T1':>9} {used_t1:>6} {ppl_t1:>9.2f} {ppl_t1/K:>7.2%}")
        print(f"{'':<6} {'':>5} {'T2':>9} {used_t2:>6} {ppl_t2:>9.2f} {ppl_t2/K:>7.2%}")
    print("=" * 88)
    print()
    print("Interpretation:")
    print("  • util → 1.0 and JSD(T1,T2) small  ⇒ codebook is saturated but not modality-specific:")
    print("                                         capacity-starved (problem 3).")
    print("  • util → 1.0 and JSD(T1,T2) large  ⇒ style is doing its job; look elsewhere.")
    print("  • util low                          ⇒ codebook under-used → not capacity; check")
    print("                                         recon/contrastive gradient balance (problem 1).")

    # ── Plot marginal usage histograms per level, overlaid by modality ──────
    n_lvls = len(style_counts)
    fig, axes = plt.subplots(1, n_lvls, figsize=(5 * n_lvls, 3.5), squeeze=False)
    for i, lvl in enumerate(sorted(style_counts.keys())):
        K = style_sizes[lvl]
        ax = axes[0, i]
        c_t1 = style_counts[lvl]["T1"]
        c_t2 = style_counts[lvl]["T2"]
        order = np.argsort(-(c_t1 + c_t2))  # sort by combined frequency
        x = np.arange(K)
        if c_t1 is not None:
            ax.bar(x - 0.2, c_t1[order] / c_t1.sum(), width=0.4, label="T1", alpha=0.8)
        if c_t2 is not None:
            ax.bar(x + 0.2, c_t2[order] / c_t2.sum(), width=0.4, label="T2", alpha=0.8)
        ax.set_title(f"Level {lvl} style codebook (K={K})")
        ax.set_xlabel("code index (sorted by total usage)")
        ax.set_ylabel("P(code)")
        ax.legend()
    plt.tight_layout()
    plt.show()

## 12-pre-b. Style-Path Utilisation Diagnostics

Three complementary checks for whether the style path is actually being used at level 0. If it isn't, modality information must flow through **content**, which explains poor L0 invariance even when style has ample (continuous) capacity.

1. **Activation norms.** Compare ‖style_L0‖ / ‖content_L0‖ over a batch of subjects. If style norm is much smaller than content norm (or near zero), style is going dead.
2. **Recon ablation.** Re-run the model with `decoders[0]`'s style input zeroed. If reconstruction barely changes, the decoder isn't conditioning on style → modality has to be in content.
3. **Decoder-0 style-path weight norm.** Inspect the weights in `decoders[0]` that consume style (FiLM conv, or the style slice of `final_conv`). Near-zero weights → decoder has learned to ignore style.


In [ ]:
# ── Diagnostic 1: Activation norms of content vs style at L0 ─────────────────
import numpy as np

content_idx = np.array(all_content_indices[0]) if 0 in all_content_indices else np.arange(hidden_channels)
style_idx_arr = np.array(all_style_indices.get(0, [])) if len(all_style_indices.get(0, [])) > 0 else None
if style_idx_arr is None or len(style_idx_arr) == 0:
    print("⚠ No style channels at level 0 — skipping style-path diagnostics.")
else:
    enc_stack = vqvae_module.encoders

    norms_content = {"T1": [], "T2": []}
    norms_style = {"T1": [], "T2": []}
    per_ch_style_means = {"T1": [], "T2": []}

    print(f"Collecting L0 activation norms over {len(items)} subjects (T1 + T2) ...")
    for i, item in enumerate(items):
        if i % 20 == 0:
            print(f"  {i}/{len(items)} ...")
        data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
        transformed = val_transforms(data_dict)
        for mod, key in [("T1", "image_t1"), ("T2", "image_t2")]:
            img = transformed[key].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                enc_out = enc_stack[0](img)  # (1, hidden_channels, D, H, W)
            feat = enc_out.squeeze(0).cpu().numpy()
            c_feat = feat[content_idx]
            s_feat = feat[style_idx_arr]
            # Per-element RMS so channel counts don't bias comparison
            norms_content[mod].append(float(np.sqrt((c_feat**2).mean())))
            norms_style[mod].append(float(np.sqrt((s_feat**2).mean())))
            per_ch_style_means[mod].append(np.abs(s_feat).mean(axis=(1, 2, 3)))  # (S,)

    print()
    print("=" * 72)
    print(f"{'':<10} {'content RMS':>14} {'style RMS':>14} {'style/content':>16}")
    print("-" * 72)
    for mod in ["T1", "T2"]:
        cm = np.mean(norms_content[mod])
        sm = np.mean(norms_style[mod])
        print(f"{mod:<10} {cm:>14.4f} {sm:>14.4f} {sm/cm:>16.4f}")
    print("=" * 72)
    print()
    print("Rule of thumb: ratio < ~0.1 suggests style is under-used; < ~0.01 suggests collapse.")

    # Per-channel style activation magnitude — detects partial/selective collapse
    ch_mean_t1 = np.mean(np.stack(per_ch_style_means["T1"]), axis=0)
    ch_mean_t2 = np.mean(np.stack(per_ch_style_means["T2"]), axis=0)
    fig, ax = plt.subplots(1, 1, figsize=(max(6, 0.3 * len(style_idx_arr)), 3.5))
    x = np.arange(len(style_idx_arr))
    ax.bar(x - 0.2, ch_mean_t1, width=0.4, label="T1", alpha=0.8)
    ax.bar(x + 0.2, ch_mean_t2, width=0.4, label="T2", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in style_idx_arr], rotation=90, fontsize=8)
    ax.set_xlabel("style channel index")
    ax.set_ylabel("mean |activation|")
    ax.set_title("L0 per-style-channel mean activation (T1 vs T2)")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Diagnostic 2: Recon ablation — zero out style into decoders[0] ──────────
# Compare recon MSE with normal style vs with style forced to zero at L0.
# If MSE barely moves, the decoder isn't using L0 style → modality must travel
# through content to explain T1 vs T2 reconstructions.

import torch.nn.functional as F
from contextlib import contextmanager


@contextmanager
def zero_style_in_decoder0():
    """Temporarily replace decoders[0].forward so style is zeroed."""
    dec0 = vqvae_module.decoders[0]
    orig_forward = dec0.forward

    def patched(x, style=None):
        if style is not None:
            style = torch.zeros_like(style)
        return orig_forward(x, style=style)

    dec0.forward = patched
    try:
        yield
    finally:
        dec0.forward = orig_forward


mse_normal = {"T1": [], "T2": []}
mse_ablated = {"T1": [], "T2": []}

# Subsample for speed — full recon passes are more expensive than encoder-only
N_ABLATION = min(40, len(items))
print(f"Running recon ablation on {N_ABLATION} subjects (T1 + T2) ...")
for i, item in enumerate(items[:N_ABLATION]):
    if i % 10 == 0:
        print(f"  {i}/{N_ABLATION} ...")
    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)
    for mod, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)
        _vi = 0 if mod == "T1" else 1
        with torch.no_grad():
            recon_n = vqvae_model(img, return_recon=True, pool_only=False, view_idx=_vi)[0]
            with zero_style_in_decoder0():
                recon_a = vqvae_model(img, return_recon=True, pool_only=False, view_idx=_vi)[0]
        if recon_n.shape != img.shape:
            recon_n = F.interpolate(recon_n, size=img.shape[2:], mode="trilinear", align_corners=False)
            recon_a = F.interpolate(recon_a, size=img.shape[2:], mode="trilinear", align_corners=False)
        mse_normal[mod].append(F.mse_loss(recon_n, img).item())
        mse_ablated[mod].append(F.mse_loss(recon_a, img).item())

print()
print("=" * 80)
print(f"{'modality':<10} {'MSE (normal)':>14} {'MSE (style=0)':>16} {'Δ abs':>12} {'Δ rel':>10}")
print("-" * 80)
for mod in ["T1", "T2"]:
    mn = float(np.mean(mse_normal[mod]))
    ma = float(np.mean(mse_ablated[mod]))
    delta = ma - mn
    rel = delta / mn if mn > 0 else float("nan")
    print(f"{mod:<10} {mn:>14.6f} {ma:>16.6f} {delta:>12.6f} {rel:>10.2%}")
print("=" * 80)
print()
print("Interpretation:")
print("  • Δ rel < ~5%    ⇒ decoder barely uses L0 style → style path is dead,")
print("                       modality must be encoded in content (problem 1 confirmed).")
print("  • Δ rel large  ⇒ decoder relies on L0 style; content-modality leakage")
print("                       is coming from another mechanism (mask routing, grad balance).")
print("  Also check: if T2 Δ ≫ T1 Δ, the style path is asymmetrically used across modalities.")

In [ ]:
# ── Diagnostic 3: Decoder-0 style-path weight norms ─────────────────────────
# Inspect how much weight the decoder places on its style input. Near-zero
# norms ⇒ decoder has learned to ignore style (posterior-collapse analogue).

dec0 = vqvae_module.decoders[0]
print(f"decoders[0].style_channels     : {dec0.style_channels}")
print(f"decoders[0].style_injection_mode: {dec0.style_injection_mode}")
print()

if dec0.style_channels == 0:
    print("No style channels on decoder 0 — nothing to inspect.")
elif dec0.film_layers is not None:
    # FiLM mode: each SpatialFiLM has a 1×1×1 conv (style_channels → 2 * feature_channels)
    print("FiLM mode — per-layer style-conv weight norms:")
    print(f"  {'layer':<10} {'conv shape':<22} {'weight L2':>12} {'mean|w|':>12} {'bias L2':>10}")
    print("  " + "-" * 72)
    all_w_norms = []
    for i, film in enumerate(dec0.film_layers):
        w = film.conv.weight.detach().cpu()
        b = film.conv.bias.detach().cpu() if film.conv.bias is not None else None
        wn = float(w.norm())
        wm = float(w.abs().mean())
        bn = float(b.norm()) if b is not None else 0.0
        all_w_norms.append(wn)
        print(f"  film_{i:<5} {str(tuple(w.shape)):<22} {wn:>12.4e} {wm:>12.4e} {bn:>10.4e}")
    print()
    print("Note: FiLM convs are initialised to zero so the decoder starts style-agnostic.")
    print("If all norms are still ~0 after training, style is being ignored.")
    if max(all_w_norms) < 1e-3:
        print("  ⚠ All FiLM weight norms are near zero — style is effectively unused.")
elif dec0.final_conv is not None:
    # Concat mode: style appears as the last `style_channels` input channels of final_conv[0]
    conv = dec0.final_conv[0]
    w = conv.weight.detach().cpu()  # (out, in, kD, kH, kW)
    total_in = w.shape[1]
    sc = dec0.style_channels
    content_slice = w[:, : total_in - sc]
    style_slice = w[:, total_in - sc :]
    content_norm = float(content_slice.norm())
    style_norm = float(style_slice.norm())
    content_mean = float(content_slice.abs().mean())
    style_mean = float(style_slice.abs().mean())
    print("Concat mode — final_conv input-channel weight norms:")
    print(f"  content-path weight L2 : {content_norm:.4e}   mean|w|: {content_mean:.4e}")
    print(f"  style-path   weight L2 : {style_norm:.4e}   mean|w|: {style_mean:.4e}")
    print(f"  style / content ratio  : {style_norm / content_norm:.4f}")
    print()
    if style_norm / content_norm < 0.05:
        print("  ⚠ Style-path weights are ≪ content-path — decoder has largely ignored style.")
    # Per-channel style weight norms — detects which style channels are being used
    per_ch = style_slice.reshape(style_slice.shape[0], sc, -1).pow(2).sum(dim=(0, 2)).sqrt().numpy()
    fig, ax = plt.subplots(1, 1, figsize=(max(6, 0.3 * sc), 3.2))
    x = np.arange(sc)
    ax.bar(x, per_ch)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in (style_idx_arr if style_idx_arr is not None else x)], rotation=90, fontsize=8)
    ax.set_xlabel("style channel (input to final_conv)")
    ax.set_ylabel("weight L2 across output ch + kernel")
    ax.set_title("decoders[0].final_conv per-style-channel weight norm")
    plt.tight_layout()
    plt.show()

## 12. Codebook Usage by Diagnosis Class

Mean codebook entry frequency per class, shown as overlaid histograms and a heatmap.
Entries that are heavily used by one class but not others indicate class-specific
structural patterns captured by that code vector.

In [ ]:
import seaborn as sns

# Use T1 only so modality doesn't confound the class comparison
t1_cb_mask = cb_modalities == "T1"
t1_cb_labels = cb_labels[t1_cb_mask]

for lvl in range(nb_levels):
    hists = cb_histograms[lvl][t1_cb_mask]

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f"Level {lvl} — Codebook Usage by Diagnosis (T1 only)", fontsize=14)

    # Mean usage histogram per class
    ax = axes[0]
    for lab_idx, lab_name in label_names.items():
        mask = t1_cb_labels == lab_idx
        mean_usage = hists[mask].mean(axis=0)
        ax.bar(np.arange(nb_entries), mean_usage, alpha=0.5, label=lab_name)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mean frequency")
    ax.set_title("Mean codebook usage")
    ax.legend()

    # Heatmap: classes x entries
    ax = axes[1]
    usage_matrix = np.zeros((len(label_names), nb_entries))
    for lab_idx in label_names:
        usage_matrix[lab_idx] = hists[t1_cb_labels == lab_idx].mean(axis=0)
    sns.heatmap(
        usage_matrix,
        ax=ax,
        cmap="viridis",
        yticklabels=[label_names[i] for i in sorted(label_names)],
        xticklabels=False,
    )
    ax.set_xlabel("Codebook entry")
    ax.set_title("Usage heatmap (class x entry)")

    plt.tight_layout()
    plt.savefig(f"codebook_usage_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Print top codes with biggest cross-class difference
    max_diff = usage_matrix.max(axis=0) - usage_matrix.min(axis=0)
    dominant_class = np.argmax(usage_matrix, axis=0)
    top_diff = np.argsort(max_diff)[::-1][:10]
    print(f"  Level {lvl} — codes with largest cross-class difference:")
    print(f"  {'Code':>6s}  {'Dominant':>8s}  {'MaxFreq':>8s}  {'MinFreq':>8s}  {'Diff':>8s}")
    for idx in top_diff:
        dom = label_names[dominant_class[idx]]
        print(
            f"  {idx:6d}  {dom:>8s}  {usage_matrix[:, idx].max():8.4f}  "
            f"{usage_matrix[:, idx].min():8.4f}  {max_diff[idx]:8.4f}"
        )
    print()

## 12a. Codebook Usage by Demographic Group & Diagnosis

Merge codebook histograms with demographic metadata from `merged_data.csv` and compare
codebook entry distributions across **gender**, **race/ethnicity**, **education**, and **diagnosis**.

In [ ]:
# ── Load demographic metadata and merge with codebook histograms ─────────────
demo_df = pd.read_csv(os.path.join(os.path.dirname(CSV_PATH), "merged_data.csv"))

# Build a lookup from Subject -> demographic info (keep first row per subject)
demo_lookup = demo_df.drop_duplicates(subset="Subject").set_index("Subject")

# The T1 entries in cb_histograms are at even indices (0, 2, 4, ...),
# one per item in `items`. Build a DataFrame aligned with the T1-filtered hists.
t1_subjects = [item["subject"] for item in items]
cb_demo = pd.DataFrame({"Subject": t1_subjects})
cb_demo = cb_demo.join(demo_lookup, on="Subject", rsuffix="_dup")

# Decode demographic columns
gender_map = {1.0: "Male", 2.0: "Female"}
race_map = {
    1: "Am Indian/Alaskan",
    2: "Asian",
    3: "Native Hawaiian/Pacific Isl",
    4: "Black",
    5: "White",
    6: "More than one",
    7: "Unknown",
}

cb_demo["Gender"] = cb_demo["PTGENDER"].map(gender_map)
# PTRACCAT can have multi-race codes like "1|5"; take first for simplicity
cb_demo["Race"] = cb_demo["PTRACCAT"].apply(
    lambda x: race_map.get(int(str(x).split("|")[0]), f"Code {x}") if pd.notna(x) else "Unknown"
)
cb_demo["Education"] = pd.cut(cb_demo["PTEDUCAT"], bins=[0, 12, 16, 25], labels=["≤12 yrs", "13-16 yrs", "17+ yrs"])

# Verify alignment: cb_demo should have exactly as many rows as T1 histograms
n_t1 = int(t1_cb_mask.sum())
assert len(cb_demo) == n_t1, f"cb_demo has {len(cb_demo)} rows but expected {n_t1} T1 samples"

print(f"Matched {cb_demo['Gender'].notna().sum()}/{len(cb_demo)} T1 samples to demographics")
print(f"\nGroup counts:")
for col in ["Group", "Gender", "Race", "Education"]:
    print(f"  {col}: {cb_demo[col].value_counts().to_dict()}")

In [ ]:
# ── Plot codebook usage differences across demographic groups & diagnosis ─────
from scipy.stats import entropy as kl_div

grouping_cols = {
    "Group": "Diagnosis",
    "Gender": "Gender",
    "Race": "Race/Ethnicity",
    "Education": "Education Level",
}

t1_hists = {lvl: cb_histograms[lvl][t1_cb_mask] for lvl in range(nb_levels)}

for lvl in range(nb_levels):
    hists = t1_hists[lvl]

    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle(
        f"Level {lvl} — Codebook Entry Distribution by Demographic Group (T1 only)", fontsize=15, fontweight="bold"
    )

    for ax, (col, title) in zip(axes.flat, grouping_cols.items()):
        groups = cb_demo[col].dropna().unique()
        groups = sorted(groups, key=str)

        # Compute mean histogram per group
        mean_hists = {}
        for g in groups:
            mask = (cb_demo[col] == g).values
            if mask.sum() < 2:
                continue
            mean_hists[g] = hists[mask].mean(axis=0)

        # Plot as overlaid bar chart
        x = np.arange(nb_entries)
        n_groups = len(mean_hists)
        width = 0.8 / max(n_groups, 1)
        for i, (g, mh) in enumerate(mean_hists.items()):
            offset = (i - n_groups / 2 + 0.5) * width
            ax.bar(x + offset, mh, width=width, alpha=0.6, label=f"{g} (n={int((cb_demo[col]==g).sum())})")

        ax.set_xlabel("Codebook entry")
        ax.set_ylabel("Mean frequency")
        ax.set_title(f"By {title}")
        ax.legend(fontsize=8, loc="upper right")

    plt.tight_layout()
    plt.savefig(f"codebook_demographic_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Difference heatmaps: for each grouping, show per-entry deviation from overall mean ──
    fig, axes = plt.subplots(2, 2, figsize=(20, 10))
    fig.suptitle(f"Level {lvl} — Codebook Usage Deviation from Overall Mean", fontsize=15, fontweight="bold")

    overall_mean = hists.mean(axis=0)

    for ax, (col, title) in zip(axes.flat, grouping_cols.items()):
        groups = sorted(cb_demo[col].dropna().unique(), key=str)
        group_names = []
        diff_matrix = []
        for g in groups:
            mask = (cb_demo[col] == g).values
            if mask.sum() < 2:
                continue
            group_names.append(f"{g} (n={int(mask.sum())})")
            diff_matrix.append(hists[mask].mean(axis=0) - overall_mean)

        if len(diff_matrix) == 0:
            ax.set_visible(False)
            continue

        diff_matrix = np.array(diff_matrix)
        vmax = np.abs(diff_matrix).max()
        sns.heatmap(
            diff_matrix,
            ax=ax,
            cmap="RdBu_r",
            center=0,
            vmin=-vmax,
            vmax=vmax,
            yticklabels=group_names,
            xticklabels=False,
        )
        ax.set_xlabel("Codebook entry")
        ax.set_title(f"By {title}")

    plt.tight_layout()
    plt.savefig(f"codebook_demographic_diff_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Print KL divergence between group pairs ─────────────────────────────────
    print(f"\n  Level {lvl} — KL divergence between group-mean distributions:")
    for col, title in grouping_cols.items():
        groups = sorted(cb_demo[col].dropna().unique(), key=str)
        mean_h = {}
        for g in groups:
            mask = (cb_demo[col] == g).values
            if mask.sum() >= 2:
                h = hists[mask].mean(axis=0)
                mean_h[g] = h / h.sum()  # re-normalize
        gnames = list(mean_h.keys())
        if len(gnames) < 2:
            continue
        print(f"    {title}:")
        for i in range(len(gnames)):
            for j in range(i + 1, len(gnames)):
                kl = kl_div(mean_h[gnames[i]] + 1e-10, mean_h[gnames[j]] + 1e-10)
                print(f"      {gnames[i]} vs {gnames[j]}: KL = {kl:.6f}")
    print()

## 12b. Dominant Code Pairs by Spatial Position

For each spatial position in the code map, we look at which codebook entries appear
most often across all T1 subjects. If the **top-2 codes** at a position both exceed
a frequency threshold (e.g., 30% of subjects), that position exhibits a **dominant
pair** — meaning the model consistently alternates between two codes at that location.

We then aggregate these pairs across all positions and levels to find the most
common dominant pairs in the dataset. This reveals:
- Whether the codebook is being used efficiently (many distinct pairs) or
  degenerately (a few codes dominate everywhere)
- Spatial structure in code assignment (e.g., do certain brain regions always
  alternate between the same two codes?)

In [ ]:
from collections import Counter

# ── Configuration ────────────────────────────────────────────────
FREQ_THRESHOLD = 0.30  # both top-2 codes must appear in ≥ this fraction of subjects
TOP_K_PAIRS = 20  # how many top pairs to show per level

print(
    f"Frequency threshold: both codes must appear in ≥ {FREQ_THRESHOLD:.0%} of "
    f"{n_t1_subjects} T1 subjects (≥ {int(FREQ_THRESHOLD * n_t1_subjects)} subjects)\n"
)

all_level_results = {}

for lvl in range(nb_levels):
    counts = pos_counts[lvl]  # (D, H, W, nb_entries), long
    spatial_shape = counts.shape[:-1]
    n_positions = counts[..., 0].numel()

    # Convert to frequencies
    freqs = counts.float() / n_t1_subjects  # (D, H, W, nb_entries)

    # Top-2 codes and their frequencies at each position
    top2_freqs, top2_codes = freqs.topk(2, dim=-1)  # each (D, H, W, 2)
    freq1 = top2_freqs[..., 0]  # most common
    freq2 = top2_freqs[..., 1]  # second most common
    code1 = top2_codes[..., 0]
    code2 = top2_codes[..., 1]

    # Positions where both top-2 codes exceed threshold
    dominant_mask = (freq1 >= FREQ_THRESHOLD) & (freq2 >= FREQ_THRESHOLD)
    n_dominant = dominant_mask.sum().item()

    print(f"{'='*65}")
    print(f"Level {lvl}  —  spatial {tuple(spatial_shape)}  " f"({n_positions} positions)")
    print(
        f"  Positions with dominant pair (both ≥ {FREQ_THRESHOLD:.0%}): "
        f"{n_dominant} / {n_positions} ({n_dominant/n_positions:.1%})"
    )

    if n_dominant == 0:
        print("  No dominant pairs found at this level. " "Try lowering FREQ_THRESHOLD.\n")
        all_level_results[lvl] = {"n_dominant": 0, "pair_counts": Counter()}
        continue

    # Extract the dominant pairs as sorted tuples (so (a,b) == (b,a))
    c1 = code1[dominant_mask].numpy()
    c2 = code2[dominant_mask].numpy()
    f1 = freq1[dominant_mask].numpy()
    f2 = freq2[dominant_mask].numpy()

    pair_counter = Counter()
    pair_freq_sum = {}  # accumulate frequencies for averaging
    for i in range(len(c1)):
        pair = tuple(sorted([int(c1[i]), int(c2[i])]))
        pair_counter[pair] += 1
        if pair not in pair_freq_sum:
            pair_freq_sum[pair] = [0.0, 0.0, 0]  # sum_freq1, sum_freq2, count
        pair_freq_sum[pair][0] += f1[i]
        pair_freq_sum[pair][1] += f2[i]
        pair_freq_sum[pair][2] += 1

    all_level_results[lvl] = {
        "n_dominant": n_dominant,
        "pair_counts": pair_counter,
        "dominant_mask": dominant_mask,
    }

    # Print top pairs
    print(f"\n  Top-{min(TOP_K_PAIRS, len(pair_counter))} dominant pairs:")
    print(f"  {'Pair':<16} {'# positions':>12} {'% of dominant':>14} " f"{'Avg freq code1':>15} {'Avg freq code2':>15}")
    print(f"  {'-'*74}")

    for pair, count in pair_counter.most_common(TOP_K_PAIRS):
        pct = count / n_dominant * 100
        s = pair_freq_sum[pair]
        avg_f1 = s[0] / s[2]
        avg_f2 = s[1] / s[2]
        print(
            f"  ({pair[0]:>3d}, {pair[1]:>3d})     {count:>8d}     {pct:>10.1f}%"
            f"     {avg_f1:>11.1%}     {avg_f2:>11.1%}"
        )

    # Summary stats
    n_unique_pairs = len(pair_counter)
    n_unique_codes = len(set(c for pair in pair_counter for c in pair))
    print(f"\n  Unique dominant pairs: {n_unique_pairs}")
    print(f"  Unique codes involved: {n_unique_codes} / {nb_entries}")

    # Distribution of how many positions each pair covers
    pair_sizes = list(pair_counter.values())
    print(
        f"  Positions per pair: min={min(pair_sizes)}, "
        f"median={sorted(pair_sizes)[len(pair_sizes)//2]}, "
        f"max={max(pair_sizes)}"
    )
    print()

# ── Cross-level summary ──────────────────────────────────────────
print("=" * 65)
print("Cross-level summary")
print("=" * 65)

# Find pairs that appear as dominant at multiple levels
all_pairs = Counter()
for lvl in range(nb_levels):
    for pair in all_level_results[lvl]["pair_counts"]:
        all_pairs[pair] += all_level_results[lvl]["pair_counts"][pair]

# Pairs that are dominant at more than one level
multi_level_pairs = []
for pair in all_pairs:
    levels_present = [lvl for lvl in range(nb_levels) if pair in all_level_results[lvl]["pair_counts"]]
    if len(levels_present) > 1:
        total = sum(all_level_results[l]["pair_counts"][pair] for l in levels_present)
        multi_level_pairs.append((pair, levels_present, total))

multi_level_pairs.sort(key=lambda x: -x[2])

if multi_level_pairs:
    print(f"\nPairs dominant at multiple levels ({len(multi_level_pairs)} pairs):")
    for pair, levels, total in multi_level_pairs[:15]:
        lvl_str = ", ".join(str(l) for l in levels)
        print(f"  ({pair[0]:>3d}, {pair[1]:>3d})  levels=[{lvl_str}]  " f"total positions={total}")
else:
    print("\nNo pairs are dominant at multiple levels.")

# Overall code utilization in dominant positions
all_dominant_codes = set()
for lvl in range(nb_levels):
    for pair in all_level_results[lvl]["pair_counts"]:
        all_dominant_codes.update(pair)
print(
    f"\nTotal unique codes in dominant pairs (all levels): "
    f"{len(all_dominant_codes)} / {nb_entries} "
    f"({len(all_dominant_codes)/nb_entries:.1%})"
)

## 13. Most Discriminative Codes — Mutual Information & Chi-squared

**Mutual information** measures how much knowing a codebook entry's usage reduces
uncertainty about the diagnosis label (higher = more informative).

**Chi-squared** tests whether the usage distribution of each entry is independent
of the class label (higher statistic = stronger association).

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import chi2_contingency

for lvl in range(nb_levels):
    hists = cb_histograms[lvl][t1_cb_mask]
    y = t1_cb_labels

    # ── Mutual Information ────────────────────────────────────────────
    mi_scores = mutual_info_classif(hists, y, discrete_features=False, random_state=42)

    fig, axes = plt.subplots(1, 2, figsize=(18, 4))
    fig.suptitle(f"Level {lvl} — Codebook Discriminativeness (T1 only)", fontsize=14)

    ax = axes[0]
    ax.bar(np.arange(nb_entries), mi_scores, color="darkorange", width=1.0)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mutual information (nats)")
    ax.set_title("MI: codebook entry usage vs diagnosis")

    top_mi = np.argsort(mi_scores)[::-1][:10]
    print(f"Level {lvl} — top-10 MI codes: {top_mi.tolist()}")
    print(f"  MI values: {mi_scores[top_mi].round(4).tolist()}")

    # ── Chi-squared ───────────────────────────────────────────────────
    chi2_scores = np.zeros(nb_entries)
    for code_idx in range(nb_entries):
        usage = hists[:, code_idx]
        # Bin non-zero usage into terciles for the contingency table
        nonzero = usage[usage > 0]
        if len(nonzero) < 10:
            continue
        bins = np.quantile(nonzero, [0.33, 0.66])
        if len(np.unique(bins)) < 2:
            continue
        digitized = np.digitize(usage, bins)
        contingency = pd.crosstab(digitized, y)
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            chi2, p, _, _ = chi2_contingency(contingency)
            chi2_scores[code_idx] = chi2

    TOP_K = 20
    top_chi2 = np.argsort(chi2_scores)[::-1][:TOP_K]

    ax = axes[1]
    ax.bar(range(TOP_K), chi2_scores[top_chi2], color="steelblue")
    ax.set_xticks(range(TOP_K))
    ax.set_xticklabels(top_chi2, rotation=45)
    ax.set_xlabel("Codebook entry index")
    ax.set_ylabel("Chi-squared statistic")
    ax.set_title(f"Top {TOP_K} most class-discriminative codes (chi2)")

    plt.tight_layout()
    plt.savefig(f"codebook_discriminativeness_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print()

## 14. PCA & t-SNE of Codebook Usage Histograms

Each subject is represented by a vector of codebook entry frequencies. If the
codebook has learned class-relevant structure, subjects with the same diagnosis
should cluster together in this histogram space.

In [ ]:
for lvl in range(nb_levels):
    hists = cb_histograms[lvl][t1_cb_mask]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(hists)
    ev = pca.explained_variance_ratio_

    tsne = TSNE(n_components=2, perplexity=min(30, len(hists) - 1), random_state=42)
    X_tsne = tsne.fit_transform(hists)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Level {lvl} — Codebook Histogram Embeddings (T1 only)", fontsize=14)

    for lab_idx, lab_name in label_names.items():
        m = t1_cb_labels == lab_idx
        axes[0].scatter(X_pca[m, 0], X_pca[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)
        axes[1].scatter(X_tsne[m, 0], X_tsne[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)

    axes[0].set_title(f"PCA ({ev.sum()*100:.1f}% var)")
    axes[0].set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    axes[0].set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")
    axes[0].legend(fontsize=8)

    axes[1].set_title("t-SNE")
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"codebook_histogram_embeddings_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

# Combined: concatenate histograms across all levels
X_all = np.concatenate([cb_histograms[lvl][t1_cb_mask] for lvl in range(nb_levels)], axis=1)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_all)
ev = pca.explained_variance_ratio_
tsne = TSNE(n_components=2, perplexity=min(30, len(X_all) - 1), random_state=42)
X_tsne = tsne.fit_transform(X_all)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("All Levels Combined — Codebook Histogram Embeddings (T1 only)", fontsize=14)
for lab_idx, lab_name in label_names.items():
    m = t1_cb_labels == lab_idx
    axes[0].scatter(X_pca[m, 0], X_pca[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)
    axes[1].scatter(X_tsne[m, 0], X_tsne[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)
axes[0].set_title(f"PCA ({ev.sum()*100:.1f}% var)")
axes[0].legend(fontsize=8)
axes[1].set_title("t-SNE")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("codebook_histogram_embeddings_all_levels.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved codebook histogram embedding plots")

## 15. Codebook Vector Replacement — CN vs AD Comparison

Replace a specific codebook entry at a chosen level with a different entry, decode
back to an image, and compare the effect on a **healthy (CN)** and **diseased (AD)**
subject side by side. This reveals whether different codes capture class-specific
structural information (e.g., atrophy patterns in AD).

The model's `decode_codes` method handles the hierarchical decoding and automatically
provides a zero-style placeholder when `inject_style_to_decoder` is active.

In [ ]:
import torch.nn.functional as F

# ── Configuration ────────────────────────────────────────────────
TARGET_LEVEL = nb_levels - 1  # coarsest level (most semantic); change to 0 for finest
SLICE_AXIS = 2  # 0=sagittal, 1=coronal, 2=axial
COMPARE_CLASSES = ["CN", "AD"]  # classes to compare side by side

# Verify both classes are available
for cls in COMPARE_CLASSES:
    assert cls in class_examples, (
        f"Class '{cls}' not found in class_examples. " f"Available: {list(class_examples.keys())}"
    )

# ── Auto-select codes to swap ───────────────────────────────────
# Use the CN example to pick codes (most-used → least-used)
cn_ids = class_examples["CN"]["id_outputs"][TARGET_LEVEL]
unique_codes, counts = cn_ids.unique(return_counts=True)
OLD_CODE = unique_codes[counts.argmax()].item()

NEW_CODE = unique_codes[counts.argmin()].item()
if NEW_CODE == OLD_CODE:
    all_present = set(unique_codes.cpu().tolist())
    absent = [c for c in range(nb_entries) if c not in all_present]
    NEW_CODE = absent[0] if absent else (OLD_CODE + 1) % nb_entries

print(f"Target level: {TARGET_LEVEL} (0=finest, {nb_levels-1}=coarsest)")
print(f"Replacing code {OLD_CODE} -> {NEW_CODE}")
print(f"Comparing classes: {COMPARE_CLASSES}\n")

# ── Embedding diagnostics ───────────────────────────────────────
codebook = vqvae_module.codebooks[TARGET_LEVEL]
emb_old = codebook.embed[:, OLD_CODE]
emb_new = codebook.embed[:, NEW_CODE]
l2_dist = (emb_old - emb_new).norm().item()
cos_sim = F.cosine_similarity(emb_old.unsqueeze(0), emb_new.unsqueeze(0)).item()
print(f"Embedding comparison (code {OLD_CODE} vs {NEW_CODE}):")
print(f"  L2 distance:       {l2_dist:.4f}")
print(f"  Cosine similarity: {cos_sim:.4f}")

all_embeds = codebook.embed.T
sample_idx = torch.randint(0, all_embeds.shape[0], (min(200, nb_entries),))
sample_embeds = all_embeds[sample_idx]
pairwise = torch.cdist(sample_embeds.unsqueeze(0), sample_embeds.unsqueeze(0)).squeeze()
tri_mask = torch.triu(torch.ones_like(pairwise, dtype=torch.bool), diagonal=1)
print(f"  Pairwise L2 stats (sample): mean={pairwise[tri_mask].mean():.4f}, " f"std={pairwise[tri_mask].std():.4f}\n")

# ── Run replacement for each class ──────────────────────────────
class_results = {}  # class_name -> dict with recon_orig, recon_modified, diff, n_replaced, image

for cls in COMPARE_CLASSES:
    ex = class_examples[cls]
    ids = ex["id_outputs"]
    img = ex["image"]
    subj = ex["item"].get("subject", "?")

    print(f"--- {cls} (subject {subj}) ---")
    print(f"  Code map shape at level {TARGET_LEVEL}: {tuple(ids[TARGET_LEVEL].shape)}")

    # Replace codes
    modified_ids = [i.clone() for i in ids]
    n_replaced = (modified_ids[TARGET_LEVEL] == OLD_CODE).sum().item()
    modified_ids[TARGET_LEVEL][modified_ids[TARGET_LEVEL] == OLD_CODE] = NEW_CODE
    print(f"  Replaced {n_replaced} voxels from code {OLD_CODE} -> {NEW_CODE}")

    with torch.no_grad():
        recon_orig = vqvae_module.decode_codes(*ids)
        recon_modified = vqvae_module.decode_codes(*modified_ids)

    # Interpolate to input spatial size if needed
    input_shape = img.shape[2:]
    if recon_orig.shape[2:] != input_shape:
        recon_orig = F.interpolate(recon_orig, size=input_shape, mode="trilinear", align_corners=False)
    if recon_modified.shape[2:] != input_shape:
        recon_modified = F.interpolate(recon_modified, size=input_shape, mode="trilinear", align_corners=False)

    # Mask background using the preprocessed (already brain-masked) input so decoder
    # activations outside the brain don't show up in the diff.
    brain_mask = (img != 0).to(recon_orig.dtype)
    recon_orig = recon_orig * brain_mask
    recon_modified = recon_modified * brain_mask

    diff = (recon_modified - recon_orig).abs()
    print(f"  Diff — max={diff.max().item():.6f}, mean={diff.mean().item():.6f}")
    rel_max = diff.max().item() / (recon_orig.abs().max().item() + 1e-8)
    print(f"  Relative max diff: {rel_max:.4%}")

    class_results[cls] = {
        "recon_orig": recon_orig,
        "recon_modified": recon_modified,
        "diff": diff,
        "n_replaced": n_replaced,
        "image": img,
        "subject": subj,
    }


# ── Visualize side by side ──────────────────────────────────────
def get_slices(vol, axis):
    """Get quarter, mid, and three-quarter slices from (C, D, H, W)."""
    n = vol.shape[axis + 1]
    return [
        vol[0].select(axis, n // 4).cpu().numpy(),
        vol[0].select(axis, n // 2).cpu().numpy(),
        vol[0].select(axis, 3 * n // 4).cpu().numpy(),
    ]


n_classes = len(COMPARE_CLASSES)
# Layout: rows = slice positions (Q1, Mid, Q3)
# For each class: Original | Recon | Modified | Diff  → 4 cols per class
n_cols = 4 * n_classes
slice_labels = ["Q1", "Mid", "Q3"]

# Compute shared intensity ranges across classes for consistent display
all_recons = [class_results[c]["recon_orig"] for c in COMPARE_CLASSES] + [
    class_results[c]["recon_modified"] for c in COMPARE_CLASSES
]
vmin_r = min(r.min().item() for r in all_recons)
vmax_r = max(r.max().item() for r in all_recons)
diff_max = max(max(class_results[c]["diff"].max().item(), 1e-8) for c in COMPARE_CLASSES)

fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 14))

for ci, cls in enumerate(COMPARE_CLASSES):
    r = class_results[cls]
    col_offset = ci * 4
    volumes = [r["image"], r["recon_orig"], r["recon_modified"], r["diff"]]
    titles = [
        f"{cls} (subj {r['subject']})\nOriginal input",
        f"{cls}\nDecoded (orig codes)",
        f"{cls}\nDecoded (modified)",
        f"{cls}\nDifference",
    ]

    for col_i, (vol, title) in enumerate(zip(volumes, titles)):
        slices = get_slices(vol[0], SLICE_AXIS)
        for row in range(3):
            ax = axes[row, col_offset + col_i]
            if "difference" in title.lower():
                im = ax.imshow(slices[row], cmap="hot", origin="lower", vmin=0, vmax=diff_max)
                if row == 0:
                    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            elif "input" in title.lower():
                ax.imshow(slices[row], cmap="gray", origin="lower")
            else:
                ax.imshow(slices[row], cmap="gray", origin="lower", vmin=vmin_r, vmax=vmax_r)
            if row == 0:
                ax.set_title(title, fontsize=10)
            ax.set_ylabel(slice_labels[row], fontsize=9) if col_i == 0 else None
            ax.axis("off")

fig.suptitle(
    f"Code Replacement — Level {TARGET_LEVEL}, code {OLD_CODE} → {NEW_CODE}\n"
    f"L2 dist={l2_dist:.4f}, cos_sim={cos_sim:.4f}, "
    f"shared diff scale max={diff_max:.6f}",
    fontsize=13,
)
plt.tight_layout()
plt.savefig("codebook_replacement_cn_vs_ad.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Summary table ───────────────────────────────────────────────
print("\nSummary:")
print(f"{'Class':<8} {'Subject':<12} {'# replaced':>12} {'Max diff':>12} {'Mean diff':>12} {'Rel max':>12}")
print("-" * 72)
for cls in COMPARE_CLASSES:
    r = class_results[cls]
    rel = r["diff"].max().item() / (r["recon_orig"].abs().max().item() + 1e-8)
    print(
        f"{cls:<8} {str(r['subject']):<12} {r['n_replaced']:>12d} "
        f"{r['diff'].max().item():>12.6f} {r['diff'].mean().item():>12.6f} {rel:>11.4%}"
    )

print("\n✓ Saved codebook_replacement_cn_vs_ad.png")

## 16. Save Reconstructions as NIfTI — CN vs AD

Save the original input, VQ-VAE reconstruction, code-modified reconstruction, and
absolute difference map as NIfTI files (`.nii.gz`) for **each class** (CN, AD).
The affine matrix is taken from MONAI's post-transform MetaTensor so the saved
volumes are in the correct preprocessed coordinate space (RAS orientation, resampled,
padded/cropped) and can be loaded directly in FSLeyes, ITK-SNAP, or 3D Slicer.

In [ ]:
import nibabel as nib

# ── Output directory ─────────────────────────────────────────────
NIFTI_OUT_DIR = "nifti_outputs"
os.makedirs(NIFTI_OUT_DIR, exist_ok=True)


# ── Helper: build fallback affine from preprocessing params ──────
def _make_fallback_affine():
    """RAS-aligned affine with the correct spacing, centred at origin."""
    D, H, W = spatial_size
    aff = np.eye(4)
    aff[0, 0] = spacing
    aff[1, 1] = spacing
    aff[2, 2] = spacing
    aff[0, 3] = -spacing * (D - 1) / 2.0
    aff[1, 3] = -spacing * (H - 1) / 2.0
    aff[2, 3] = -spacing * (W - 1) / 2.0
    return aff


# ── Helper: save tensor → .nii.gz ───────────────────────────────
def save_nifti(tensor, affine, filepath, description=""):
    """Save a torch tensor as a compressed NIfTI file.

    Args:
        tensor: (1, 1, D, H, W) or (1, D, H, W) or (D, H, W) torch tensor.
        affine: 4x4 numpy affine matrix.
        filepath: Output path (should end in .nii.gz).
        description: Optional description for the NIfTI header.
    """
    vol = tensor.detach().cpu().float().numpy()
    while vol.ndim > 3:
        vol = vol[0]
    img = nib.Nifti1Image(vol, affine)
    if description:
        img.header["descrip"] = description.encode("ascii", errors="ignore")[:80]
    nib.save(img, filepath)
    print(f"  Saved: {filepath}  {vol.shape}  [{vol.min():.4f}, {vol.max():.4f}]")


# ── Save volumes for each class ──────────────────────────────────
for cls in COMPARE_CLASSES:
    ex = class_examples[cls]
    r = class_results[cls]
    subject_id = ex["item"].get("subject", "unknown")

    # Use MetaTensor affine if available, else fallback
    affine = ex["affine"].copy() if ex["affine"] is not None else _make_fallback_affine()
    affine_src = "MetaTensor" if ex["affine"] is not None else "constructed"

    print(f"\n{'='*60}")
    print(f"{cls} — subject {subject_id}  (affine: {affine_src})")
    print(f"{'='*60}")

    prefix = f"{NIFTI_OUT_DIR}/{cls}_subj_{subject_id}"

    save_nifti(
        r["image"],
        affine,
        f"{prefix}_input.nii.gz",
        description=f"{cls} VQ-VAE input (preprocessed T1)",
    )

    save_nifti(
        r["recon_orig"],
        affine,
        f"{prefix}_recon_orig.nii.gz",
        description=f"{cls} VQ-VAE reconstruction (original codes)",
    )

    save_nifti(
        r["recon_modified"],
        affine,
        f"{prefix}_recon_modified_L{TARGET_LEVEL}_c{OLD_CODE}to{NEW_CODE}.nii.gz",
        description=f"{cls} recon (L{TARGET_LEVEL}: {OLD_CODE}->{NEW_CODE})",
    )

    save_nifti(
        r["diff"],
        affine,
        f"{prefix}_diff_L{TARGET_LEVEL}_c{OLD_CODE}to{NEW_CODE}.nii.gz",
        description=f"{cls} diff (L{TARGET_LEVEL}: {OLD_CODE}->{NEW_CODE})",
    )

print(f"\n✓ All NIfTI volumes saved to {NIFTI_OUT_DIR}/")
print("  Open in FSLeyes / ITK-SNAP / 3D Slicer for interactive inspection.")
print(f"  Affine matches preprocessed space (RAS, {spacing}mm, {spatial_size}).")
print(f"\n  Files per class:")
for cls in COMPARE_CLASSES:
    subj = class_examples[cls]["item"].get("subject", "unknown")
    print(f"    {cls} (subj {subj}): *input, *recon_orig, *recon_modified, *diff")

## 17. Demographic Probes — Sex & Race on Content vs Style

Train **linear** (logistic regression) and **non-linear** (MLP) probes to predict
**sex** and **race** from content-only, style-only, and concatenated features at each
level plus aggregates. Uses T1 scans only so modality does not confound the comparison.

If content captures shared anatomy, demographic traits visible in anatomy (sex) should
be predictable from content. If style captures modality-specific contrast, demographics
should ideally not be encoded there — any strong signal is leakage.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# DEMOGRAPHIC PROBES — sex & race on content vs style features
# Linear (logistic regression) + non-linear (MLP) classifiers, T1 only
# to avoid modality confound. Reuses all_features, level_content,
# all_subjects, all_modalities, _per_view_select, nb_levels.
# ══════════════════════════════════════════════════════════════════
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.dummy import DummyClassifier

# ── Load demographics and align with all_subjects / all_modalities ───────────
_demo_df = pd.read_csv(os.path.join(os.path.dirname(CSV_PATH), "merged_data.csv"))
_demo_lookup = _demo_df.drop_duplicates(subset="Subject").set_index("Subject")

_gender_map = {1.0: "Male", 2.0: "Female"}
_race_map = {
    1: "Am Indian/Alaskan",
    2: "Asian",
    3: "Native Hawaiian/Pacific Isl",
    4: "Black",
    5: "White",
    6: "More than one",
    7: "Unknown",
}


def _lookup_gender(subj):
    if subj in _demo_lookup.index:
        v = _demo_lookup.loc[subj, "PTGENDER"]
        return _gender_map.get(float(v)) if pd.notna(v) else None
    return None


def _lookup_race(subj):
    if subj in _demo_lookup.index:
        v = _demo_lookup.loc[subj, "PTRACCAT"]
        if pd.isna(v):
            return None
        first = str(v).split("|")[0]
        try:
            return _race_map.get(int(first))
        except ValueError:
            return None
    return None


_sex_all = np.array([_lookup_gender(s) for s in all_subjects], dtype=object)
_race_all = np.array([_lookup_race(s) for s in all_subjects], dtype=object)

# T1 only + drop missing demographics
_t1_mask = all_modalities == "T1"

# ── Feature sets (per-level content/style + aggregates) ──────────────────────
_content_parts, _style_parts = [], []
_feature_sets_all = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    if lc["content_idx"]:
        X_c = _per_view_select(feats, lc["content_idx"], lc["content_idx_v1"])
        _feature_sets_all[f"Content L{lvl}"] = X_c
        _content_parts.append(X_c)
    if lc["style_idx"]:
        X_s = _per_view_select(feats, lc["style_idx"], lc["style_idx_v1"])
        _feature_sets_all[f"Style L{lvl}"] = X_s
        _style_parts.append(X_s)
    _feature_sets_all[f"All L{lvl}"] = feats
if _content_parts:
    _feature_sets_all["Content (all levels)"] = np.concatenate(_content_parts, axis=1)
if _style_parts:
    _feature_sets_all["Style (all levels)"] = np.concatenate(_style_parts, axis=1)
if _content_parts and _style_parts:
    _feature_sets_all["Content ⊕ Style"] = np.concatenate(
        [_feature_sets_all["Content (all levels)"], _feature_sets_all["Style (all levels)"]], axis=1
    )


# ── Probe factories ──────────────────────────────────────────────────────────
def _lr_probe():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, C=1.0, random_state=42, class_weight="balanced"),
    )


def _mlp_probe():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            max_iter=500,
            early_stopping=True,
            random_state=42,
        ),
    )


_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Run probes for each target ───────────────────────────────────────────────
_targets = {"Sex": _sex_all, "Race": _race_all}
demographic_probe_results = {}

for target_name, y_all in _targets.items():
    valid = _t1_mask & np.array([v is not None for v in y_all])
    y = y_all[valid].astype(str)
    classes, counts = np.unique(y, return_counts=True)
    # Drop rare classes (< 2*n_splits) that would break stratified CV
    keep_classes = classes[counts >= 2 * _cv.n_splits]
    mask_keep = np.isin(y, keep_classes)
    y_str = y[mask_keep]
    valid_idx = np.where(valid)[0][mask_keep]

    # MLPClassifier's early-stopping scorer calls np.isnan on predictions,
    # which fails on object/string labels. Encode to ints.
    _le = LabelEncoder()
    y = _le.fit_transform(y_str)
    class_names = list(_le.classes_)

    n_classes = len(np.unique(y))
    chance = cross_val_score(DummyClassifier(strategy="most_frequent"), np.zeros((len(y), 1)), y, cv=_cv).mean()

    print("=" * 86)
    print(f"DEMOGRAPHIC PROBE — Target: {target_name}")
    print(f"N = {len(y)} (T1 only, after dropping missing / rare classes)")
    _labels, _cnts = np.unique(y_str, return_counts=True)
    print(f"Classes ({n_classes}): {dict(zip(_labels, _cnts))}")
    print(f"Majority-class baseline = {chance*100:.1f}%")
    print("=" * 86)
    print(f"{'Feature set':<25} {'Dim':>5} {'LR balAcc':>14} {'MLP balAcc':>14}")
    print("-" * 86)

    per_target = {}
    for name, X_full in _feature_sets_all.items():
        X = X_full[valid_idx]
        lr_sc = cross_val_score(_lr_probe(), X, y, cv=_cv, scoring="balanced_accuracy")
        mlp_sc = cross_val_score(_mlp_probe(), X, y, cv=_cv, scoring="balanced_accuracy")
        per_target[name] = {
            "lr": lr_sc.mean(),
            "lr_std": lr_sc.std(),
            "mlp": mlp_sc.mean(),
            "mlp_std": mlp_sc.std(),
            "dim": X.shape[1],
        }
        print(
            f"{name:<25} {X.shape[1]:>5d} "
            f"{lr_sc.mean()*100:>8.1f}% ±{lr_sc.std()*100:>3.1f} "
            f"{mlp_sc.mean()*100:>8.1f}% ±{mlp_sc.std()*100:>3.1f}"
        )
    per_target["_chance"] = chance
    demographic_probe_results[target_name] = per_target
    print()

# ── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(_targets), figsize=(8 * len(_targets), 0.35 * len(_feature_sets_all) + 2))
if len(_targets) == 1:
    axes = [axes]

for ax, (target_name, per_target) in zip(axes, demographic_probe_results.items()):
    chance = per_target.pop("_chance")
    names = list(per_target.keys())
    lr_vals = [per_target[n]["lr"] * 100 for n in names]
    mlp_vals = [per_target[n]["mlp"] * 100 for n in names]
    lr_err = [per_target[n]["lr_std"] * 100 for n in names]
    mlp_err = [per_target[n]["mlp_std"] * 100 for n in names]
    y_pos = np.arange(len(names))
    h = 0.38
    ax.barh(y_pos - h / 2, lr_vals, h, xerr=lr_err, label="Linear (LR)", color="tab:blue", alpha=0.85)
    ax.barh(y_pos + h / 2, mlp_vals, h, xerr=mlp_err, label="Non-linear (MLP)", color="tab:orange", alpha=0.85)
    ax.axvline(chance * 100, color="red", ls="--", label=f"Majority ({chance*100:.1f}%)")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel("Balanced accuracy (%)")
    ax.set_title(f"Predict {target_name}")
    ax.set_xlim(0, 100)
    ax.legend(loc="lower right", fontsize=8)
    # restore chance for downstream inspection
    per_target["_chance"] = chance

plt.suptitle("Demographic Probes — Content vs Style (T1 only, 5-fold CV)", fontsize=13)
plt.tight_layout()
plt.savefig("demographic_probes_sex_race.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Saved demographic_probes_sex_race.png")
print("  Results available in `demographic_probe_results` dict.")

## 18. L0 Leakage Diagnostics

Three checks to determine whether low `sep_L0` is **real modality leakage** at L0 or just **metric asymmetry** (anatomy info naturally lives at coarser levels).

1. **Modality probe per level** — directly measures how much T1-vs-T2 info each content space carries. Pure leakage signal.
2. **Normalized separation** — `(anatomy − modality) / (full_anatomy − chance)` per level, to quotient out each level's anatomy headroom.
3. **L0-ablation proxy** — compare anatomy accuracy using `Content L0 only` vs `Content L1+L2 only` vs `Content all`. If L0-only is near chance while L1+L2 ≈ full, L0 content isn't carrying anatomy.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# L0 LEAKAGE DIAGNOSTICS
# Requires cell 69 to have run (uses _feature_sets_all, _targets,
# demographic_probe_results, _t1_mask, _cv, _lr_probe, level_content,
# all_features, all_modalities, nb_levels, _per_view_select, _sex_all,
# _race_all).
# ══════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier

_cv_all = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def _probe_balAcc(X, y):
    """Mean balanced accuracy across 5-fold CV for linear probe on X, y."""
    return cross_val_score(_lr_probe(), X, y, cv=_cv_all, scoring="balanced_accuracy").mean()


# ── Diagnostic 1: modality probe per level (T1+T2 rows, labelled by view) ───
modality_y = (all_modalities == "T2").astype(int)
modality_chance = cross_val_score(
    DummyClassifier(strategy="most_frequent"),
    np.zeros((len(modality_y), 1)),
    modality_y,
    cv=_cv_all,
    scoring="balanced_accuracy",
).mean()

modality_probe_per_level = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    content_X = _per_view_select(feats, lc["content_idx"], lc["content_idx_v1"]) if lc["content_idx"] else None
    style_X = _per_view_select(feats, lc["style_idx"], lc["style_idx_v1"]) if lc["style_idx"] else None

    row = {"chance": modality_chance}
    if content_X is not None:
        row["content"] = _probe_balAcc(content_X, modality_y)
    if style_X is not None:
        row["style"] = _probe_balAcc(style_X, modality_y)
    row["all"] = _probe_balAcc(feats, modality_y)
    modality_probe_per_level[lvl] = row

print("=" * 78)
print("DIAGNOSTIC 1 — Modality probe per level (T1 vs T2, balanced accuracy)")
print(f"Chance (majority-class) baseline = {modality_chance*100:.1f}%")
print("=" * 78)
print(f"{'Level':<8}{'content':>12}{'style':>12}{'all':>12}{'leakage':>14}")
print("-" * 78)
for lvl, row in modality_probe_per_level.items():
    leakage = row.get("content", np.nan) - modality_chance
    print(
        f"L{lvl:<7}"
        f"{row.get('content', float('nan'))*100:>11.1f}%"
        f"{row.get('style', float('nan'))*100:>11.1f}%"
        f"{row.get('all', float('nan'))*100:>11.1f}%"
        f"{leakage*100:>+13.1f}pp"
    )
print(
    "  → content column above chance = modality info leaking into content. "
    "Compare L0 vs L1/L2: if L0 is noticeably higher, leakage is real."
)


# ── Diagnostic 2: normalized separation per level ────────────────────────────
# sep_norm_L = (mean_anatomy_acc - modality_acc) / (full_anatomy_acc - chance)
# Uses existing demographic_probe_results from cell 69 for anatomy (Sex, Race).
# Falls back to recomputing if a target is missing.
print()
print("=" * 78)
print("DIAGNOSTIC 2 — Normalized separation per level (anatomy vs modality)")
print("=" * 78)
print(f"{'Level':<8}{'anat(content)':>16}{'anat(full)':>14}{'mod(content)':>16}{'sep':>10}{'sep_norm':>12}")
print("-" * 78)

sep_table = {}
for lvl in range(nb_levels):
    anat_content = []
    anat_full = []
    for target_name in _targets:  # Sex, Race
        per = demographic_probe_results.get(target_name, {})
        content_key = f"Content L{lvl}"
        all_key = f"All L{lvl}"
        chance_key = per.get("_chance", np.nan)
        if content_key in per:
            anat_content.append(per[content_key]["lr"])
        if all_key in per:
            anat_full.append(per[all_key]["lr"])
    if not anat_content or not anat_full:
        continue
    anat_content_mean = float(np.mean(anat_content))
    anat_full_mean = float(np.mean(anat_full))
    anat_chance_mean = float(np.mean([demographic_probe_results[t]["_chance"] for t in _targets]))
    mod_content = modality_probe_per_level[lvl].get("content", np.nan)

    sep = anat_content_mean - mod_content
    headroom = anat_full_mean - anat_chance_mean
    sep_norm = sep / headroom if headroom > 1e-6 else np.nan
    sep_table[lvl] = {
        "anat_content": anat_content_mean,
        "anat_full": anat_full_mean,
        "mod_content": mod_content,
        "sep": sep,
        "sep_norm": sep_norm,
    }
    print(
        f"L{lvl:<7}"
        f"{anat_content_mean*100:>15.1f}%"
        f"{anat_full_mean*100:>13.1f}%"
        f"{mod_content*100:>15.1f}%"
        f"{sep*100:>+9.1f}"
        f"{sep_norm:>12.2f}"
    )
print(
    "  → if L0 sep_norm is close to L1/L2 sep_norm, the raw L0 sep gap is "
    "just anatomy headroom — not real leakage. If L0 sep_norm is much lower, "
    "L0 is genuinely leaking."
)


# ── Diagnostic 3: L0-ablation proxy ──────────────────────────────────────────
# Compare anatomy probe accuracy for (a) L0-only content, (b) L1+L2-only
# content, (c) all-levels content. If L0-only ≈ chance and L1+L2 ≈ all, then
# L0 content isn't carrying anatomy signal — consistent with "L0 content is
# mostly modality".
print()
print("=" * 78)
print("DIAGNOSTIC 3 — L0-ablation proxy (anatomy balanced accuracy per scope)")
print("=" * 78)

# Build the three feature sets on T1-only rows.
_content_by_level = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    if lc["content_idx"]:
        _content_by_level[lvl] = _per_view_select(all_features[f"level_{lvl}"], lc["content_idx"], lc["content_idx_v1"])

_scopes = {}
if 0 in _content_by_level:
    _scopes["L0 only"] = _content_by_level[0]
_higher = [_content_by_level[l] for l in range(1, nb_levels) if l in _content_by_level]
if _higher:
    _scopes["L1+L2 only"] = np.concatenate(_higher, axis=1)
if _content_by_level:
    _scopes["all levels"] = np.concatenate([_content_by_level[l] for l in sorted(_content_by_level)], axis=1)

ablation_results = {}
for target_name, y_all in _targets.items():
    valid = _t1_mask & np.array([v is not None for v in y_all])
    y = y_all[valid].astype(str)
    classes, counts = np.unique(y, return_counts=True)
    keep = classes[counts >= 2 * _cv_all.n_splits]
    mask_keep = np.isin(y, keep)
    y_str = y[mask_keep]
    valid_idx = np.where(valid)[0][mask_keep]
    y_enc = LabelEncoder().fit_transform(y_str)

    chance = cross_val_score(
        DummyClassifier(strategy="most_frequent"),
        np.zeros((len(y_enc), 1)),
        y_enc,
        cv=_cv_all,
        scoring="balanced_accuracy",
    ).mean()

    print(f"\nTarget: {target_name}  (chance = {chance*100:.1f}%)")
    print(f"  {'Scope':<14}{'balAcc':>10}{'Δ vs chance':>16}")
    print("  " + "-" * 40)
    row = {"_chance": chance}
    for scope_name, X_full in _scopes.items():
        X = X_full[valid_idx]
        acc = _probe_balAcc(X, y_enc)
        row[scope_name] = acc
        print(f"  {scope_name:<14}{acc*100:>9.1f}%{(acc-chance)*100:>+15.1f}pp")
    ablation_results[target_name] = row

print(
    "\n  → L0-only near chance + L1+L2 ≈ all-levels  ⇒  L0 content is mostly "
    "noise/modality; upper levels carry anatomy. Consider shrinking C_content_L0."
)


# ── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))

# (a) Modality probe per level — leakage bar
ax = axes[0]
lvls = sorted(modality_probe_per_level)
content_vals = [modality_probe_per_level[l].get("content", np.nan) * 100 for l in lvls]
style_vals = [modality_probe_per_level[l].get("style", np.nan) * 100 for l in lvls]
x = np.arange(len(lvls))
w = 0.35
ax.bar(x - w / 2, content_vals, w, label="content", color="tab:blue")
ax.bar(x + w / 2, style_vals, w, label="style", color="tab:orange")
ax.axhline(modality_chance * 100, color="red", ls="--", label=f"chance ({modality_chance*100:.1f}%)")
ax.set_xticks(x)
ax.set_xticklabels([f"L{l}" for l in lvls])
ax.set_ylabel("Modality balAcc (%)")
ax.set_title("Diagnostic 1 — modality probe per level")
ax.legend()

# (b) sep vs sep_norm per level
ax = axes[1]
lvls2 = sorted(sep_table)
sep_vals = [sep_table[l]["sep"] * 100 for l in lvls2]
sep_norm_vals = [sep_table[l]["sep_norm"] * 100 for l in lvls2]
x = np.arange(len(lvls2))
ax.bar(x - w / 2, sep_vals, w, label="raw sep", color="tab:blue")
ax.bar(x + w / 2, sep_norm_vals, w, label="sep_norm ×100", color="tab:green")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"L{l}" for l in lvls2])
ax.set_ylabel("%")
ax.set_title("Diagnostic 2 — raw vs normalized separation")
ax.legend()

# (c) L0 ablation proxy
ax = axes[2]
targets_plot = list(ablation_results.keys())
scopes = [k for k in _scopes]
x = np.arange(len(targets_plot))
w = 0.8 / max(len(scopes), 1)
for i, scope in enumerate(scopes):
    vals = [ablation_results[t].get(scope, np.nan) * 100 for t in targets_plot]
    ax.bar(x + (i - (len(scopes) - 1) / 2) * w, vals, w, label=scope)
for i, t in enumerate(targets_plot):
    ax.hlines(ablation_results[t]["_chance"] * 100, i - 0.4, i + 0.4, colors="red", linestyles="dashed")
ax.set_xticks(x)
ax.set_xticklabels(targets_plot)
ax.set_ylabel("Anatomy balAcc (%)")
ax.set_title("Diagnostic 3 — L0-ablation proxy\n(red dashes = chance)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("l0_leakage_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✓ Saved l0_leakage_diagnostics.png")
print("  Results: `modality_probe_per_level`, `sep_table`, `ablation_results`")